# Flood Research v18: Reproducible Outputs, Execution Guards, and Teknika SINTA-2 Readiness

**Target journal:** *Teknika: Journal of Information and Communication Technology* (IKADO, SINTA-2).  
**Base dataset:** Mendeley Data v1, DOI `10.17632/d7j3s64vvr.1`, covering 2020–2025.  
**Primary estimand:** classification of a region-day with a recorded flood report, not verified physical flood occurrence.  
**Core method:** temporal out-of-fold stacking, validation-only probability calibration, and paired missingness stress tests.

## What this revision changes

1. Rebuilds the model pipeline with a new model-version signature and rejects mixed evidence files from different runs.
2. Keeps training (2020–2023), calibration/threshold selection (2024), and the repeatedly inspected 2025 retrospective evaluation separate.
3. Increases missingness stress simulations to 100 replicates per mechanism and level, while distinguishing simulation spread from inferential confidence intervals.
4. Saves the fitted model, calibrators, feature schema, thresholds, and provenance outside the publication-output folder.
5. Supports an optional temporal holdout from 2026 onward when updated source workbooks are supplied through `SOURCE_OVERRIDE_DIR`. The model still trains only on 2020–2023 and calibrates only on 2024.
6. Exports exactly 10 PNG figures and 10 CSV tables into one ZIP. The final cell calls the official Python API `google.colab.files.download()` from a verified local `/content` copy. It never calls a free-standing JavaScript `download()` function.
7. Separates the model/evidence-cache version from the notebook/output version so that publication packaging does not silently force a different model specification.
7. Pins the model-cache signature to the source hashes, feature-engineering version, estimator settings, OOF stress settings, calibration rule, and model-pipeline version.
8. Clears only the dedicated publication-output directory before generating artifacts, then validates file counts, image integrity, DPI, and ZIP contents.

## Non-negotiable scientific limitation

The original 2025 data have influenced earlier development iterations. Re-running this notebook cannot turn 2025 into an untouched confirmatory test. If no post-2025 data are supplied, the results remain exploratory and the editorial gate will explicitly report that independent validation is missing. Even with later data, the model predicts recorded reports unless independently verified flood-event labels and reliable issue-time metadata support a stronger claim.

**Run order:** run all cells from top to bottom. Keep `FAST_TEST = False` for publication-grade outputs. The first configuration cell contains `SOURCE_OVERRIDE_DIR`. For optional 2026+ evaluation, place all four updated source workbooks in one folder and set that path before running. Do not use later outcomes to tune the model.

**Important:** code quality and journal fit do not guarantee Q1 acceptance. The target journal's current quartile should be checked in the relevant ranking source at submission time. The notebook deliberately labels 2025 results as exploratory because earlier versions already inspected them.

**Journal-fit note (checked 18 October 2026):** Teknika's scope includes AI, machine learning, data mining, and ICT applications. The manuscript must be written in English, follow the journal template, use a 6–10 page A4 two-column layout, keep the abstract within 250 words, and use up to five keywords. The notebook can audit evidence and reproducibility, but it cannot certify the manuscript template, similarity score, independent validation, or acceptance.


**Execution limitation:** the notebook must be run top-to-bottom in Colab to regenerate model metrics and all plots. The accompanying ZIP contains the latest available output artifacts, with editorial checks updated, and is not evidence that the entire v18 pipeline has already completed in Colab.


In [ ]:
# Memasang hanya paket yang belum tersedia; semua parameter penelitian dikunci di sini.
import sys, subprocess, importlib.util
pkgs = {'numpy':'numpy','pandas':'pandas','matplotlib':'matplotlib',
        'sklearn':'scikit-learn','openpyxl':'openpyxl','requests':'requests','joblib':'joblib'}
missing = [v for k,v in pkgs.items() if importlib.util.find_spec(k) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])

import os, io, re, json, time, hashlib, shutil, zipfile, unicodedata
from pathlib import Path
from datetime import datetime, date, timezone
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import requests, joblib
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import (average_precision_score,roc_auc_score,brier_score_loss,
    log_loss,f1_score,precision_score,recall_score,precision_recall_curve,roc_curve)
from IPython.display import display

PIPELINE_VERSION = 'v8.0.0-locked-temporal-missingness-prospective-audit-20261018'
# Versi model v8 memisahkan cache dari semua versi sebelumnya; konfigurasi eksperimen dikunci untuk reprodusibilitas.
EVIDENCE_AUDIT_VERSION = 'v15.0-locked-temporal-prospective-validation-audit'
SEED = 42
RNG = np.random.default_rng(SEED)
FAST_TEST = False           # Hanya True untuk uji teknis, bukan untuk angka publikasi.
FORCE_RETRAIN = False       # True untuk memaksa fitting ulang walau cache dengan signature identik tersedia.
BOOTSTRAP_REPS = 1000       # Bootstrap berpasangan berbasis blok tanggal 7 hari.
ROBUSTNESS_REPS = 100       # Replikasi per skenario untuk mengurangi ketergantungan pada satu seed.
MISSING_LEVELS = [0.00, 0.10, 0.20, 0.30]
BLOCK_LENGTH_DAYS = 7

def sample_non_circular_blocks(n_days, block_days, rng):
    """Resample chronological blocks without wrapping Dec 31 to Jan 1."""
    n_days, block_days = int(n_days), int(block_days)
    if n_days < 1 or block_days < 1 or block_days > n_days:
        raise ValueError(f'Invalid block bootstrap dimensions: n_days={n_days}, block_days={block_days}')
    n_blocks = int(np.ceil(n_days / block_days))
    starts = rng.integers(0, n_days - block_days + 1, size=n_blocks)
    return np.concatenate([np.arange(start, start + block_days) for start in starts])[:n_days]
STRESS_LEVELS_OOF = [0.10,0.20,0.30,0.20]    # Nilai eksploratif untuk v5; test 2025 telah dilihat pada v3/v4.
STRESS_METHODS_OOF = ['MCAR','Station outage','Temporal block','Quality-dependent']
STRESS_CLEAN_WEIGHT = 0.50             # Rasio eksploratif; tidak diklaim preregistered.
FEATURE_ENGINEERING_VERSION = 'lagged-region-rainfall-v3'
CALIBRATION_RULE = 'logistic_calibration_on_logit_probability_fitted_on_2024'
THRESHOLD_RULE = 'max_f1_on_2024_apparent_validation_only'
MODEL_SPEC_VERSION = 'LR-RF-HGB-logit-meta-v8-quality-dependent-OOF' 

IN_COLAB = Path('/content').exists()
PERSIST_TO_DRIVE = IN_COLAB
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ROOT = Path('/content/drive/MyDrive/Flood_Research_Mendeley_v1')
else:
    ROOT = (Path('/mnt/data/Flood_Research_Mendeley_v1') if Path('/mnt/data').exists() else Path.cwd() / 'Flood_Research_Mendeley_v1')

SOURCES = ROOT / 'source'
CACHE = ROOT / 'cache'
OUTPUT = ROOT / 'v15_training_evidence_cache'
SOURCE_OVERRIDE_DIR = ''  # Opsional: folder empat workbook versi diperbarui yang mencakup data 2026+.
ATTEST_FUTURE_DATA_UNSEEN_DURING_MODEL_DEVELOPMENT = False  # Ubah True hanya jika benar-benar dapat dipertanggungjawabkan.
for d in [ROOT,SOURCES,CACHE,OUTPUT]: d.mkdir(parents=True,exist_ok=True)

REQUIRED_FILES = [
    'Data-Ch-Jabar-2020-2025.xlsx',
    'Dataset_Flood_2020-2025.xlsx',
    'Dataset_Rainfall_Floods_2020-2025.xlsx',
    'West_Java_Rainfall_Station_Metadata_Lengkap.xlsx'
]
DATA_URL='https://data.mendeley.com/public-api/zip/d7j3s64vvr/download/1'
COLOR={'blue':'#0072B2','orange':'#E69F00','green':'#009E73',
       'red':'#D55E00','purple':'#CC79A7','sky':'#56B4E9','black':'#222222'}
MODEL_STYLE={'Logistic Regression':('blue','o','-'),
             'Random Forest':('orange','s','--'),
             'HistGradientBoosting':('green','^','-.'),
             'Standard stacking':('purple','D','-'),
             'Full uncertainty–quality stacking':('red','P','-'),
             'Seasonal-region baseline':('black','x',':'),
             'Stress-trained uncertainty–quality stacking':('sky','v','--')}
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'axes.labelsize':10,
    'xtick.labelsize':8,'ytick.labelsize':8,'legend.fontsize':8,
    'figure.facecolor':'white','axes.spines.top':False,'axes.spines.right':False,
    'savefig.facecolor':'white','savefig.dpi':300})
print('Project:', ROOT)
print('Data source DOI: 10.17632/d7j3s64vvr.1')
print('Pipeline version:', PIPELINE_VERSION)
# Default v15 membangun ulang bukti dari workbook sumber; tidak menggunakan evidence cache lama secara otomatis.
PIPELINE_MODE = 'retrain'   # 'auto' | 'retrain' | 'audit_only'
# retrain = rebuild evidence from source; exact-signature model cache may be reused.
# Set FORCE_RETRAIN=True below only when you intentionally require a fresh estimator fit.
USER_EVIDENCE_FOLDER = ''  # Folder yang berisi tepat satu set 10 CSV v8 asli.
USER_EVIDENCE_ZIP = ''     # ZIP berisi 10 CSV v8 (opsional).
STRICT_HISTORICAL_SHA = '' # Opsional: isi hash SHA-256 set CSV bila ingin mengunci satu eksperimen.
EVIDENCE_STEMS = {
 'station':'01_station_mapping_audit','quality':'02_region_year_quality',
 'panel':'03_lagged_feature_panel','split':'04_temporal_split_balance',
 'metrics':'05_model_metrics_validation_and_test','pred':'06_held_out_test_predictions',
 'calibration':'07_test_reliability_bins','robustness':'08_station_missingness_robustness',
 'alerts':'09_operational_alert_budgets','bootstrap':'10_paired_block_bootstrap'}

In [ ]:
# v18 preflight: fail early on invalid configuration; do not silently downgrade publication mode.
assert PIPELINE_MODE in {'auto','retrain','audit_only'}, f'Invalid PIPELINE_MODE: {PIPELINE_MODE}'
assert isinstance(FAST_TEST, bool), 'FAST_TEST must be a boolean.'
if FAST_TEST:
    print('WARNING: FAST_TEST is diagnostic only; final publication export is blocked.')
assert len(set(MISSING_LEVELS)) == len(MISSING_LEVELS) and all(0 <= float(x) < 1 for x in MISSING_LEVELS), \
    'MISSING_LEVELS must be unique proportions in [0, 1).'
assert BOOTSTRAP_REPS >= 100 and ROBUSTNESS_REPS >= 20, \
    'Publication run requires at least 100 bootstrap reps and 20 robustness reps; use larger counts for final inference.'
assert len(REQUIRED_FILES) == 4 and len(set(REQUIRED_FILES)) == 4, 'Source file manifest is inconsistent.'
print('v18 preflight passed | pipeline mode:', PIPELINE_MODE, '| FAST_TEST:', FAST_TEST,
      '| bootstrap reps:', BOOTSTRAP_REPS, '| robustness reps:', ROBUSTNESS_REPS)


## A. Evidence-cache selection and integrity gate

`PIPELINE_MODE='retrain'` is the default. It rebuilds the evidence tables from the source workbooks and uses a fitted-model cache only when its signature matches the exact data hashes and model specification. `PIPELINE_MODE='auto'` may reuse a complete historical evidence set after integrity checks. `PIPELINE_MODE='audit_only'` requires a compatible evidence set and never fits a model.

The full run requires a non-fast configuration. A fast smoke test is useful for debugging only and cannot create publication-grade output. The audit rejects mixed CSV sets, duplicate region-day rows, inconsistent labels, invalid probabilities, incomplete bootstrap runs, and mismatched model signatures. The original 2025 period remains exploratory because earlier development versions already inspected it.


In [ ]:
# Cari sepuluh tabel yang konsisten; jangan menggabungkan berkas dari folder atau eksperimen berbeda.
import re, hashlib, zipfile, shutil, json
from pathlib import Path

def find_evidence_set(folder):
    """Pilih satu set kanonis; jangan mencampur file bernomor dari eksperimen berbeda."""
    folder=Path(folder)
    if not folder.is_dir(): return None
    found={}
    for key,stem in EVIDENCE_STEMS.items():
        exact=folder/(stem+'.csv')
        if exact.exists():
            found[key]=exact
            continue
        options=[]
        for f in folder.glob(stem+'*.csv'):
            m=re.fullmatch(re.escape(stem)+r'(?:\((\d+)\))?\.csv',f.name)
            if m: options.append(f)
        # File duplikat ambigu ditolak, kecuali nama kanonis tersedia.
        if len(options)!=1: return None
        found[key]=options[0]
    # Jika beberapa tabel membawa signature atau versi yang sama, nilai yang berbeda menandakan set campuran.
    signatures=set(); versions=set()
    for path in found.values():
        try:
            head=pd.read_csv(path,nrows=5)
            if 'source_signature' in head:
                signatures.update(head['source_signature'].dropna().astype(str).unique().tolist())
            if 'pipeline_version' in head:
                versions.update(head['pipeline_version'].dropna().astype(str).unique().tolist())
        except Exception:
            return None
    if len(signatures)>1 or len(versions)>1: return None
    return found

def evidence_fullrun_checks(files, require_full=True):
    try:
        import pandas as pd, numpy as np
        panel_check=pd.read_csv(files['panel'],usecols=['date','region_key','flood_recorded'])
        pred_check=pd.read_csv(files['pred'],usecols=['date','region_key','flood_recorded',
                         'p_stress_trained_uncertainty_quality_stacking'])
        robustness_check=pd.read_csv(files['robustness'],usecols=['mechanism','replicate'])
        bs_check=pd.read_csv(files['bootstrap'])
        panel_check['date']=pd.to_datetime(panel_check['date'],errors='raise')
        historical=panel_check[panel_check['date'].dt.year.le(2025)]
        assert len(historical)==59184 and int(historical.flood_recorded.sum())==1358
        assert len(pred_check)==9855 and int(pred_check.flood_recorded.sum())==296
        assert panel_check[['date','region_key']].duplicated().sum()==0
        assert pred_check[['date','region_key']].duplicated().sum()==0
        assert np.isfinite(pred_check.p_stress_trained_uncertainty_quality_stacking).all()
        if require_full:
            disturbed=robustness_check[robustness_check.mechanism.ne('Original')]
            assert disturbed.replicate.nunique()>=20, 'Only fast-test missingness replicates found'
            assert 'bootstrap_reps_successful' in bs_check.columns
            assert bs_check.bootstrap_reps_successful.max()>=900, 'Primary bootstrap has too few repetitions'
            assert bs_check.bootstrap_reps_successful.min()>=200, 'Sensitivity bootstrap has too few repetitions'
        return True,'complete full evidence' if require_full else 'valid technical evidence'
    except Exception as ex:
        return False,repr(ex)

candidate_folders=[]
if USER_EVIDENCE_FOLDER: candidate_folders.append(Path(USER_EVIDENCE_FOLDER))
candidate_folders.extend([ROOT/'v12_training_evidence_cache',ROOT/'results_v8_frozen_evidence',
                          ROOT/'v11_evidence_cache',Path('/mnt/data'),Path.cwd()])
ACTIVE_EVIDENCE_FILES=None
if PIPELINE_MODE not in ('auto','retrain','audit_only'):
    raise ValueError('PIPELINE_MODE must be auto, retrain, or audit_only')
if PIPELINE_MODE!='retrain':
    for candidate in candidate_folders:
        matched=find_evidence_set(candidate)
        if matched:
            valid,reason=evidence_fullrun_checks(matched,require_full=not FAST_TEST)
            if valid:
                ACTIVE_EVIDENCE_FILES=matched
                print('Reusing verified evidence:',candidate)
                break
            print('Rejecting incompatible / fast cache:',candidate,reason)

if ACTIVE_EVIDENCE_FILES is None and USER_EVIDENCE_ZIP and PIPELINE_MODE!='retrain':
    archive=Path(USER_EVIDENCE_ZIP)
    if not archive.exists(): raise FileNotFoundError(archive)
    extracted=ROOT/'v12_uploaded_evidence';extracted.mkdir(parents=True,exist_ok=True)
    for f in extracted.glob('*.csv'): f.unlink()
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            if not member.is_dir() and member.filename.lower().endswith('.csv'):
                dest=extracted/Path(member.filename).name
                with z.open(member) as src,dest.open('wb') as out: shutil.copyfileobj(src,out)
    candidate=find_evidence_set(extracted)
    if candidate:
        valid,reason=evidence_fullrun_checks(candidate,require_full=not FAST_TEST)
        if valid:ACTIVE_EVIDENCE_FILES=candidate
        else:raise RuntimeError('ZIP is not a full verified evidence run: '+reason)
TRAIN_REQUIRED=(PIPELINE_MODE=='retrain' or ACTIVE_EVIDENCE_FILES is None)
if PIPELINE_MODE=='audit_only' and TRAIN_REQUIRED:
    raise RuntimeError('Full v8 evidence is required in audit_only mode; set USER_EVIDENCE_FOLDER or ZIP.')
print('Training required:',TRAIN_REQUIRED,'| FAST_TEST:',FAST_TEST)

## B. Frozen model training (runs only if complete evidence is unavailable)

The sequence below is adapted from the validated v8 research pipeline, preserving the frozen v6 base-model and meta-model settings. Model training uses **2020–2023** with chronology-respecting out-of-fold meta-features; **2024** is reserved for calibration and threshold selection; **2025** is evaluated retrospectively only. Four XLSX files are acquired from Mendeley Data v1, cached on Drive, and read without deriving non-event labels from the event-conditioned consensus workbook. No stochastic missingness is applied after learning to affect the saved 2025 labels.

**Important:** A trained model produced here is an exploratory reconstruction of the already-reviewed method. A new independent outcome period would be required for confirmatory evaluation.

### B5. Source acquisition and SHA-256 provenance

In [ ]:
if TRAIN_REQUIRED:
    # Berusaha menggunakan sumber yang sudah disimpan lebih dahulu.
    # Dahulukan XLSX lokal agar unduhan HTTP tidak dipanggil berulang jika sumber sudah tersedia.
    override_dir=Path(SOURCE_OVERRIDE_DIR).expanduser() if SOURCE_OVERRIDE_DIR else None
    for source_name in REQUIRED_FILES:
        target=SOURCES/source_name
        # Workbook override selalu diprioritaskan agar data 2026+ tidak tertutup cache Mendeley v1 lama.
        if override_dir is not None and (override_dir/source_name).exists():
            shutil.copy2(override_dir/source_name,target)
        elif not target.exists():
            for location in (Path('/mnt/data')/source_name,Path.cwd()/source_name):
                if location.exists():
                    shutil.copy2(location,target)
                    break
    archive_path = SOURCES / 'mendeley_d7j3s64vvr_v1.zip'
    need_files=[name for name in REQUIRED_FILES if not (SOURCES/name).exists()]
    if need_files:
        if not archive_path.exists():
            try:
                resp=requests.get(DATA_URL, timeout=150, headers={'User-Agent':'Mozilla/5.0'})
                resp.raise_for_status()
                if resp.content[:2]!=b'PK':
                    raise ValueError('Mendeley tidak mengirim arsip ZIP.')
                archive_path.write_bytes(resp.content)
                print('ZIP sumber berhasil diunduh dan dicache.')
            except Exception as error:
                print('Akses langsung gagal:', str(error))
                if IN_COLAB:
                    from google.colab import files
                    print('Unggah ZIP asli dataset Mendeley v1 satu kali:')
                    uploaded=files.upload()
                    choices=[n for n in uploaded if n.lower().endswith('.zip')]
                    if not choices: raise FileNotFoundError('ZIP sumber tidak tersedia.')
                    archive_path.write_bytes(uploaded[choices[0]])
                else:
                    # Editor lokal dapat menggunakan file ZIP atau Excel di folder kerja.
                    local_zips=[p for p in [Path.cwd()/'mendeley_d7j3s64vvr_v1.zip',Path('/mnt/data/mendeley_d7j3s64vvr_v1.zip')] if p.exists()]
                    local_xlsx=[n for n in need_files if (Path.cwd()/n).exists() or (Path('/mnt/data')/n).exists()]
                    for n in local_xlsx: shutil.copy2((Path.cwd()/n if (Path.cwd()/n).exists() else Path('/mnt/data')/n),SOURCES/n)
                    if local_zips:
                        shutil.copy2(local_zips[0], archive_path)
                    elif any(not (SOURCES/n).exists() for n in REQUIRED_FILES):
                        raise RuntimeError('Simpan ZIP Mendeley versi 1 atau empat XLSX di /mnt/data / source.')
        if any(not (SOURCES/n).exists() for n in REQUIRED_FILES):
            with zipfile.ZipFile(archive_path) as zf:
                members={Path(info.filename).name:info for info in zf.infolist() if not info.is_dir()}
                for name in REQUIRED_FILES:
                    if (SOURCES/name).exists(): continue
                    if name not in members: raise FileNotFoundError('Berkas wajib tidak ditemukan: '+name)
                    # Ekstraksi berdasarkan nama diketahui; tidak menggunakan path dari arsip.
                    with zf.open(members[name]) as src, open(SOURCES/name,'wb') as dst:
                        shutil.copyfileobj(src,dst)

    assert all((SOURCES/n).exists() for n in REQUIRED_FILES), 'Sumber belum lengkap.'
    def sha256(path):
        dig=hashlib.sha256()
        with open(path,'rb') as f:
            for chunk in iter(lambda:f.read(2**20), b''): dig.update(chunk)
        return dig.hexdigest()
    SOURCE_SIGNATURE={n:sha256(SOURCES/n) for n in REQUIRED_FILES}
    CONFIG={'version':PIPELINE_VERSION,'seed':SEED,'fast_test':FAST_TEST,
            'bootstrap':BOOTSTRAP_REPS,'robustness_reps':ROBUSTNESS_REPS,
            'levels':MISSING_LEVELS,'source_override_dir':str(override_dir) if override_dir else '', 'source_hashes':SOURCE_SIGNATURE}
    SIGNATURE=hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:16]
    # Cache model ditentukan hanya oleh data, kode model, dan seed;
    # perubahan jumlah bootstrap/replikasi robustness tidak memaksa training ulang.
    # Signature model mencakup semua pilihan yang mengubah estimator atau konstruksi meta-data.
    # Perubahan pada fitur, fold, atau stress augmentation harus mengubah versi/spec ini.
    MODEL_CONFIG={
        'version':PIPELINE_VERSION,
        'model_spec':MODEL_SPEC_VERSION,
        'seed':SEED,
        'fast_test':FAST_TEST,
        'source_hashes':SOURCE_SIGNATURE,
        'feature_engineering_version':FEATURE_ENGINEERING_VERSION,
        'calibration_rule':CALIBRATION_RULE,
        'threshold_rule':THRESHOLD_RULE,
        'base_estimators':{
            'logistic_regression':{'C':0.5,'max_iter':1200,'class_weight':'balanced'},
            'random_forest':{'n_estimators':125,'max_depth':13,'min_samples_leaf':9,
                             'max_features':'sqrt','class_weight':'balanced_subsample'},
            'hist_gradient_boosting':{'max_iter':135,'learning_rate':0.06,'max_leaf_nodes':15,
                                      'min_samples_leaf':35,'l2_regularization':3.0}},
        'meta_estimator':{'C':0.2,'max_iter':1500,'class':'LogisticRegression'},
        'oof_windows':[('2022-01-01','2022-07-01'),('2022-07-01','2023-01-01'),
                       ('2023-01-01','2023-07-01'),('2023-07-01','2024-01-01')],
        'stress_oof_methods':STRESS_METHODS_OOF,
        'stress_oof_levels':STRESS_LEVELS_OOF,
        'stress_clean_weight':STRESS_CLEAN_WEIGHT
    }
    MODEL_SIGNATURE=hashlib.sha256(json.dumps(MODEL_CONFIG,sort_keys=True).encode()).hexdigest()[:16]
    MODEL_CACHE=CACHE/f'models_{MODEL_SIGNATURE}.joblib'
    print('Source and experiment signature:',SIGNATURE)
    for n in REQUIRED_FILES: print(n,'sha256',SOURCE_SIGNATURE[n][:16])

### B7. Date parsing, administrative mapping, and station metadata

In [ ]:
if TRAIN_REQUIRED:
    # Parser tanggal mempertahankan datetime asli, angka serial Excel, dan string campuran.
    def parse_dates(values):
        s=pd.Series(values).reset_index(drop=True)
        out=pd.Series(pd.NaT,index=s.index,dtype='datetime64[ns]')
        isdt=s.map(lambda x:isinstance(x,(pd.Timestamp,datetime,date,np.datetime64)))
        if isdt.any(): out.loc[isdt]=pd.to_datetime(s.loc[isdt],errors='coerce').to_numpy()
        rest=s.notna() & ~isdt
        if rest.any():
            numeric=pd.to_numeric(s.loc[rest],errors='coerce')
            serial=numeric.between(35000,65000)
            if serial.any():
                idx=numeric.index[serial]
                out.loc[idx]=pd.to_datetime(numeric.loc[idx],unit='D',origin='1899-12-30',errors='coerce').to_numpy()
            idx=numeric.index[~serial]
            if len(idx):
                out.loc[idx]=pd.to_datetime(s.loc[idx].astype(str),errors='coerce',format='mixed',dayfirst=False).to_numpy()
        return out.dt.normalize()

    def clean_space(value):
        t=unicodedata.normalize('NFKC',str(value)).strip().casefold()
        return re.sub(r'\s+',' ',t)

    def district_key(value,kind):
        # Pembedaan kabupaten dan kota tidak boleh dihapus saat proses join.
        raw=clean_space(value)
        if kind=='metadata':
            if re.match(r'^kota\b',raw): unit='city'; stem=re.sub(r'^kota\b','',raw).strip()
            elif re.match(r'^(kab\.|kabupaten|kab)\b',raw): unit='regency'; stem=re.sub(r'^(kab\.|kabupaten|kab)\b','',raw).strip()
            else: raise ValueError('District metadata tidak dikenal: '+str(value))
        elif kind=='flood':
            if raw.endswith('_city'):unit='city';stem=raw[:-5]
            elif raw=='west_bandung':unit='regency';stem='bandung barat'
            else:unit='regency';stem=raw
        else:raise ValueError('Sumber wilayah tidak dikenal')
        stem=re.sub(r'[^a-z0-9]+','',stem)
        return unit+'_'+stem

    rain_raw=pd.read_excel(SOURCES/REQUIRED_FILES[0],sheet_name='DATA')
    flood_raw=pd.read_excel(SOURCES/REQUIRED_FILES[1],sheet_name='Rawdata')
    metadata=pd.read_excel(SOURCES/REQUIRED_FILES[3],sheet_name='metadata')
    assert {'Date'}.issubset(rain_raw.columns)
    assert {'District','Rainfall_station','Latitude','Longitude','Elev ( m )'}.issubset(metadata.columns)
    assert {'Event_Date','District/City'}.issubset(flood_raw.columns)

    rain_raw['Date']=parse_dates(rain_raw['Date']).values
    flood_raw['Event_Date']=parse_dates(flood_raw['Event_Date']).values
    assert rain_raw['Date'].notna().all() and flood_raw['Event_Date'].notna().all(), 'Tanggal invalid.'
    assert rain_raw['Date'].is_unique, 'Tanggal hujan duplikat.'
    assert rain_raw['Date'].min()==pd.Timestamp('2020-01-01')
    assert rain_raw['Date'].max()>=pd.Timestamp('2025-12-31'), 'Data hujan harus mencakup setidaknya sampai 31 Desember 2025.'
    expected_dates=pd.date_range(rain_raw['Date'].min(),rain_raw['Date'].max(),freq='D')
    actual_dates=pd.DatetimeIndex(rain_raw['Date']).sort_values()
    assert actual_dates.equals(expected_dates), 'Tanggal pada matriks hujan harus berurutan harian tanpa celah.'
    if rain_raw['Date'].max()==pd.Timestamp('2025-12-31'):
        assert len(rain_raw)==2192 and len(flood_raw)==4231, 'Jumlah sumber berbeda dari Mendeley Data v1 yang diaudit.'
    else:
        assert len(rain_raw)>2192, 'Ekstensi temporal harus menambahkan observasi setelah 2025.'
        assert flood_raw['Event_Date'].min()>=pd.Timestamp('2020-01-01')
        assert flood_raw['Event_Date'].max()<=rain_raw['Date'].max(), 'Ada laporan banjir di luar cakupan data hujan.'

    metadata['region_key']=metadata['District'].map(lambda x:district_key(x,'metadata'))
    metadata['station_key']=metadata['Rainfall_station'].map(clean_space)
    collision=metadata.groupby('station_key')['region_key'].nunique()
    if (collision>1).any():
        raise ValueError('Metadata memiliki nama stasiun identik pada wilayah berbeda: '+str(collision[collision>1].index.tolist()))
    station_lookup=metadata.drop_duplicates('station_key').set_index('station_key')['region_key'].to_dict()
    station_cols=list(rain_raw.columns[1:])
    station_regions={st:station_lookup.get(clean_space(st)) for st in station_cols}
    station_audit=metadata[['Station_Id','Rainfall_station','District','region_key',
        'Latitude','Longitude','Elev ( m )','Missing_Data (%)']].copy()
    station_audit=station_audit.rename(columns={'Station_Id':'station_id','Rainfall_station':'station',
        'District':'district','Latitude':'latitude','Longitude':'longitude',
        'Elev ( m )':'elevation_m','Missing_Data (%)':'metadata_missing_pct'})
    station_audit['in_rainfall_matrix']=station_audit.station.isin(station_cols)
    unresolved=[st for st,region in station_regions.items() if region is None]
    for st in unresolved:
        station_audit.loc[len(station_audit)]={'station_id':'','station':st,'district':'UNMAPPED',
            'region_key':'UNMAPPED','latitude':np.nan,'longitude':np.nan,
            'elevation_m':np.nan,'metadata_missing_pct':np.nan,'in_rainfall_matrix':True}
    print('Rainfall stations:',len(station_cols),'| mapped:',len(station_cols)-len(unresolved),'| unresolved:',unresolved)
    assert len(unresolved)<=2,'Terlalu banyak stasiun tidak terpetakan.'

    flood_raw['region_key']=flood_raw['District/City'].map(lambda x:district_key(x,'flood'))
    assert set(flood_raw.region_key)==set(metadata.region_key),'Pemetaan seluruh 27 kota/kabupaten harus cocok.'
    reports=(flood_raw.groupby(['Event_Date','region_key'],as_index=False).size()
             .rename(columns={'Event_Date':'date','size':'reported_flood_events'}))
    if rain_raw['Date'].max()==pd.Timestamp('2025-12-31'):
        assert len(reports)==1358,'Jumlah wilayah-tanggal positif berbeda dari audit sumber v1.'
    else:
        assert len(reports)>=1358,'Ekstensi data tidak boleh menghilangkan catatan historis versi 1.'
    print('Unique flood region-days:',len(reports),'| source reports:',len(flood_raw))

### B9. Lagged rainfall features and region-day labels

In [ ]:
if TRAIN_REQUIRED:
    # Bangun matriks hari × stasiun; pertahankan NaN alami dan hindari pengisian nilai dari masa depan.
    kept_cols=[st for st in station_cols if station_regions[st] is not None]
    raw_matrix=(rain_raw.set_index('Date')[kept_cols]
                .apply(pd.to_numeric,errors='coerce').sort_index())
    if (raw_matrix.lt(0) & raw_matrix.notna()).any().any():
        raise ValueError('Curah hujan negatif ditemukan; periksa kode missing di data mentah.')
    if raw_matrix.max().max()>1000:
        raise ValueError('Curah hujan >1000 mm/hari: audit satuan sebelum melanjutkan.')
    region_series=pd.Series({st:station_regions[st] for st in kept_cols})
    station_count=region_series.value_counts().sort_index()

    # Fungsi fitur diulang identik pada kondisi asli maupun data hujan yang dimasking.
    def make_features(matrix):
        # Persentase observasi tersedia dihitung langsung dari matriks stasiun mentah.
        grouping=region_series.to_dict()
        t=matrix.T
        grp=t.groupby(grouping,sort=True)
        avg=grp.mean().T
        high=grp.max().T
        count=grp.count().T
        coverage=count.div(station_count,axis=1)
        dates=avg.index
        frames=[]
        for region in avg.columns:
            v=pd.DataFrame(index=dates)
            mean=avg[region]
            high_r=high[region]
            cov=coverage[region]
            for lag in [1,2,3,7]:
                v[f'rain_mean_lag{lag}_mm']=mean.shift(lag)
                v[f'rain_max_lag{lag}_mm']=high_r.shift(lag)
            for window in [3,7]:
                v[f'rain_sum_prev{window}_mm']=mean.shift(1).rolling(window,min_periods=window).sum()
                v[f'coverage_prev{window}_mean']=cov.shift(1).rolling(window,min_periods=window).mean()
                # Nilai tanpa observasi tetap NaN, tidak dihitung sebagai hari kering.
                prior_rain=mean.shift(1)
                wet_indicator=prior_rain.gt(1.0).where(prior_rain.notna(),np.nan)
                v[f'rain_wetdays_prev{window}']=wet_indicator.rolling(window,min_periods=1).sum().where(
                    wet_indicator.rolling(window,min_periods=1).count().gt(0),np.nan)
                v[f'rain_observed_days_prev{window}']=prior_rain.notna().astype(float).rolling(
                    window,min_periods=window).sum()
                # Menghitung hari hujan hanya di antara hari yang benar-benar memiliki pengamatan.
                v[f'rain_wet_fraction_prev{window}']=v[f'rain_wetdays_prev{window}']/v[f'rain_observed_days_prev{window}'].replace(0,np.nan)
            v['coverage_lag1']=cov.shift(1)
            v['observed_stations_lag1']=count[region].shift(1)
            v['expected_station_count']=int(station_count[region])
            v['month_sin']=np.sin(2*np.pi*(dates.month-1)/12)
            v['month_cos']=np.cos(2*np.pi*(dates.month-1)/12)
            v['region_key']=region
            v.index.name='date'
            frames.append(v.reset_index())
        result=pd.concat(frames,ignore_index=True).sort_values(['date','region_key']).reset_index(drop=True)
        return result

    panel=make_features(raw_matrix)
    panel=panel.merge(reports,on=['date','region_key'],how='left',validate='one_to_one')
    panel['reported_flood_events']=panel['reported_flood_events'].fillna(0).astype('int16')
    panel['flood_recorded']=panel['reported_flood_events'].gt(0).astype('int8')
    panel['outcome_definition']='recorded flood vs no recorded flood'
    panel['year']=panel.date.dt.year
    panel['data_partition']=np.select(
        [panel.year.between(2020,2023),panel.year.eq(2024),panel.year.eq(2025),panel.year.ge(2026)],
        ['training','calibration_validation','exploratory_test_2025','prospective_holdout_2026plus'],
        default='outside')
    # Panel harus lengkap pada setiap kombinasi tanggal dan wilayah; label positif harus sama dengan laporan yang terpetakan.
    expected_region_days=len(raw_matrix.index)*len(station_count)
    assert len(panel)==expected_region_days, 'Panel region-day tidak lengkap.'
    assert panel[['date','region_key']].duplicated().sum()==0, 'Duplikasi region-day ditemukan.'
    assert panel.date.nunique()==len(raw_matrix.index), 'Jumlah tanggal pada panel berbeda dari matriks hujan.'
    assert int(panel.flood_recorded.sum())==len(reports), 'Label panel tidak konsisten dengan catatan wilayah-hari dari laporan banjir.'
    historical=panel[panel.year.le(2025)]
    assert len(historical)==59184 and int(historical.flood_recorded.sum())==1358, 'Jendela historis 2020–2025 berbeda dari sumber v1.'

    FEATURE_NUMERIC=[c for c in panel.columns if c.startswith(('rain_','coverage_','observed_','expected_','month_'))]
    DUMMIES=[f'region_{k}' for k in sorted(panel.region_key.unique())]
    region_dummies=pd.get_dummies(panel.region_key,prefix='region',dtype='int8')
    X_all=pd.concat([panel[FEATURE_NUMERIC].reset_index(drop=True),region_dummies.reset_index(drop=True)],axis=1)
    FEATURE_COLUMNS=X_all.columns.tolist()
    QUALITY_COLUMNS=['coverage_lag1','coverage_prev3_mean','coverage_prev7_mean','expected_station_count']
    TRAIN_MASK=panel.year.between(2020,2023).to_numpy()
    VAL_MASK=panel.year.eq(2024).to_numpy()
    TEST_MASK=panel.year.eq(2025).to_numpy()
    PROSPECTIVE_MASK=panel.year.ge(2026).to_numpy()
    y_all=panel.flood_recorded.to_numpy(dtype=int)
    assert all(np.unique(y_all[mask]).size==2 for mask in [TRAIN_MASK,VAL_MASK,TEST_MASK])
    assert X_all.loc[TEST_MASK,FEATURE_COLUMNS].shape[0]==9855
    if PROSPECTIVE_MASK.any():
        assert panel.loc[PROSPECTIVE_MASK,'date'].dt.year.min()>=2026
        print('Prospective rows available:',int(PROSPECTIVE_MASK.sum()),
              '| distinct dates:',panel.loc[PROSPECTIVE_MASK,'date'].nunique(),
              '| positive reports:',int(y_all[PROSPECTIVE_MASK].sum()))

    # Validasi: seluruh fitur hujan berasal dari tanggal sebelum label target.
    for checkcol in ['rain_mean_lag1_mm','rain_mean_lag7_mm','coverage_lag1']:
        assert checkcol in FEATURE_NUMERIC
    # Unit test: hari hilang tidak berubah menjadi hari kering; tidak ada akses hujan target.
    _test_wet=pd.Series([2.0,np.nan,0.0,4.0])
    _w=_test_wet.gt(1.0).where(_test_wet.notna(),np.nan)
    assert np.isnan(_w.iloc[1]), 'Hari tanpa observasi tidak boleh dinyatakan kering.'
    assert int(_w.notna().sum())==3
    assert _w.iloc[0] == 1 and _w.iloc[2] == 0, 'Indikator basah/kering berubah tidak semestinya.'
    print('Region-days:',len(panel),'| labels:',int(y_all.sum()),'| predictors:',len(FEATURE_COLUMNS))
    print(panel.groupby('data_partition').flood_recorded.agg(['size','sum','mean']).to_string())

### B11. Temporal OOF training of frozen base/meta learners

In [ ]:
if TRAIN_REQUIRED:
    # Menentukan algoritme dasar yang sama untuk seluruh varian stacking.
    def make_base_models():
        return {
            'Logistic Regression':Pipeline([
                ('imp',SimpleImputer(strategy='median',add_indicator=False)),
                ('scale',StandardScaler()),
                ('clf',LogisticRegression(C=0.5,max_iter=1200,class_weight='balanced',random_state=SEED))]),
            'Random Forest':Pipeline([
                ('imp',SimpleImputer(strategy='median')),
                ('clf',RandomForestClassifier(n_estimators=125,max_depth=13,
                    min_samples_leaf=9,max_features='sqrt',class_weight='balanced_subsample',
                    n_jobs=-1,random_state=SEED))]),
            'HistGradientBoosting':Pipeline([
                ('imp',SimpleImputer(strategy='median')),
                ('clf',HistGradientBoostingClassifier(max_iter=135,learning_rate=.06,max_leaf_nodes=15,
                    min_samples_leaf=35,l2_regularization=3.0,random_state=SEED))])}

    BASE_NAMES=list(make_base_models())
    META_NAMES=['Standard stacking','Uncertainty-only stacking',
                'Quality-only stacking','Full uncertainty–quality stacking',
                'Stress-trained uncertainty–quality stacking']
    ALL_NAMES=BASE_NAMES+META_NAMES+['Seasonal-region baseline']
    META_SETS={'Standard stacking':BASE_NAMES,
        'Uncertainty-only stacking':BASE_NAMES+['model_disagreement','mean_predictive_entropy'],
        'Quality-only stacking':BASE_NAMES+QUALITY_COLUMNS,
        'Full uncertainty–quality stacking':BASE_NAMES+['model_disagreement','mean_predictive_entropy']+QUALITY_COLUMNS,
        'Stress-trained uncertainty–quality stacking':BASE_NAMES+['model_disagreement','mean_predictive_entropy']+QUALITY_COLUMNS}

    # Mean probability entropy dan perbedaan antarmodel dipakai sebagai proksi ketidakpastian.
    def make_meta_features(prob_arr,X):
        prob_arr=np.clip(np.asarray(prob_arr),1e-6,1-1e-6)
        out=pd.DataFrame(prob_arr,columns=BASE_NAMES)
        out['model_disagreement']=prob_arr.std(axis=1)
        out['mean_predictive_entropy']=(-prob_arr*np.log(prob_arr)-
            (1-prob_arr)*np.log(1-prob_arr)).mean(axis=1)
        for col in QUALITY_COLUMNS:
            out[col]=np.asarray(X[col],dtype=float)
        return out

    training_idx=np.flatnonzero(TRAIN_MASK)
    val_idx=np.flatnonzero(VAL_MASK)
    test_idx=np.flatnonzero(TEST_MASK)
    future_idx=np.flatnonzero(PROSPECTIVE_MASK)
    train_dates=panel.loc[TRAIN_MASK,'date'].reset_index(drop=True)
    X_train=X_all.iloc[training_idx].reset_index(drop=True)
    y_train=y_all[training_idx]
    X_val=X_all.iloc[val_idx].reset_index(drop=True)
    y_val=y_all[val_idx]
    X_test=X_all.iloc[test_idx].reset_index(drop=True)
    y_test=y_all[test_idx]
    X_future=X_all.iloc[future_idx].reset_index(drop=True)
    y_future=y_all[future_idx]

    windows=[('2022-01-01','2022-07-01'),('2022-07-01','2023-01-01'),
             ('2023-01-01','2023-07-01'),('2023-07-01','2024-01-01')]
    folds=[]
    for begin,end in windows:
        fit=np.flatnonzero(train_dates.lt(begin))
        hold=np.flatnonzero(train_dates.ge(begin)&train_dates.lt(end))
        assert len(fit)>0 and len(hold)>0
        assert y_train[fit].sum()>0 and y_train[hold].sum()>0
        assert train_dates.iloc[fit].max() < train_dates.iloc[hold].min()
        folds.append((fit,hold))


    # Augmentasi meta-learner menggunakan prediksi out-of-fold yang terkena kehilangan data stasiun.
    # Fitur model dasar dihitung ulang SETELAH stasiun dihilangkan, bukan dengan meniadakan fitur agregat.
    # Tidak ada label atau model tahun 2025 yang digunakan untuk pelatihan.
    def stress_mask_for_oof(base, mechanism, level, seed):
        rng=np.random.default_rng(seed)
        x=base.astype(float).copy()  # Paksa float agar nilai NaN tidak ditulis ke kolom integer.
        arr=x.to_numpy(dtype=float,copy=True)
        available=np.isfinite(arr)
        mask=np.zeros_like(available,dtype=bool)
        if mechanism=='MCAR':
            mask=rng.random(arr.shape)<level
        elif mechanism=='Station outage':
            n=max(1,int(round(arr.shape[1]*level)))
            mask[:,rng.choice(arr.shape[1],size=n,replace=False)]=True
        elif mechanism=='Quality-dependent':
            # Bobot berasal hanya dari TRAIN historis yang mendahului fold OOF.
            # Gunakan periode tahun 2020 sampai akhir data fit, tanpa melihat tahun 2025.
            prior=raw_matrix.loc[:pd.Timestamp(base.index.min())-pd.Timedelta(days=1)]
            if len(prior)==0:prior=raw_matrix.loc['2020-01-01':'2021-12-31']
            hist_missing=prior.isna().mean().reindex(base.columns).fillna(0).to_numpy()
            weights=.1+hist_missing
            weights=weights/max(float(weights.mean()),1e-12)
            mask=rng.random(arr.shape)<np.minimum(level*weights[None,:],1.0)
        elif mechanism=='Temporal block':
            target=int(round(level*available.sum()))
            newly=0
            for _ in range(200000):
                if newly>=target:break
                j=int(rng.integers(arr.shape[1]))
                start=int(rng.integers(max(1,arr.shape[0]-BLOCK_LENGTH_DAYS+1)))
                stop=min(arr.shape[0],start+BLOCK_LENGTH_DAYS)
                add=int(np.count_nonzero(available[start:stop,j] & ~mask[start:stop,j]))
                mask[start:stop,j]=True
                newly+=add
        else:
            raise ValueError('Unknown training missingness mechanism')
        arr[mask & available]=np.nan
        x.iloc[:,:]=arr
        return x

    def fold_stress_meta(fold_models, hold, begin, end, mechanism, level, seed):
        # Jendela 14 hari sebelum hold memberi konteks fitur lag+rolling tanpa data masa depan.
        start=pd.Timestamp(begin)-pd.Timedelta(days=14)
        finish=pd.Timestamp(end)-pd.Timedelta(days=1)
        src=raw_matrix.loc[start:finish].copy()
        damaged=stress_mask_for_oof(src,mechanism,level,seed)
        computed=make_features(damaged)
        computed=computed[(computed.date>=begin)&(computed.date<end)].reset_index(drop=True)
        wanted=panel.iloc[training_idx[hold]][['date','region_key']].reset_index(drop=True)
        assert computed[['date','region_key']].equals(wanted), 'OOF augmented observations misaligned.'
        Xdamaged=X_train.iloc[hold].reset_index(drop=True).copy()
        Xdamaged[FEATURE_NUMERIC]=Xdamaged[FEATURE_NUMERIC].astype(float)
        for col in FEATURE_NUMERIC:
            Xdamaged[col]=computed[col].to_numpy(dtype=float)
        base_p=np.column_stack([fold_models[name].predict_proba(Xdamaged)[:,1] for name in BASE_NAMES])
        return make_meta_features(base_p,Xdamaged)

    # Cache harus dapat dibaca dan membawa signature model yang sama; cache rusak tidak boleh menghentikan pipeline.
    CACHE_VALID=False
    cached_state=None
    if MODEL_CACHE.exists() and not FORCE_RETRAIN:
        try:
            cached_state=joblib.load(MODEL_CACHE)
            CACHE_VALID=(
                isinstance(cached_state,dict)
                and cached_state.get('model_signature')==MODEL_SIGNATURE
                and cached_state.get('model_spec_version')==MODEL_SPEC_VERSION
                and {'base_models','meta_models','oof_valid_rows','stress_augmentation_rows'}.issubset(cached_state)
            )
            if not CACHE_VALID:
                print('Cached model metadata does not match current signature; a fresh fit will be performed.')
        except Exception as cache_error:
            print('Cached model could not be loaded; a fresh fit will be performed:',repr(cache_error))
            CACHE_VALID=False
    if CACHE_VALID:
        state=cached_state
        print('Loaded exact-signature cached models:',MODEL_CACHE)
    else:
        if MODEL_CACHE.exists() and FORCE_RETRAIN:
            print('FORCE_RETRAIN=True: ignoring cached estimators.')
        oof=np.full((len(X_train),len(BASE_NAMES)),np.nan)
        fold_predictors=[]
        for foldnum,(fit,hold) in enumerate(folds,start=1):
            print(f'OOF {foldnum}/{len(folds)}: fit={len(fit):,}, future holdout={len(hold):,}')
            fmodels={}
            for j,(name,estimator) in enumerate(make_base_models().items()):
                fitted=clone(estimator).fit(X_train.iloc[fit],y_train[fit])
                oof[hold,j]=fitted.predict_proba(X_train.iloc[hold])[:,1]
                fmodels[name]=fitted
            fold_predictors.append(fmodels)
        valid=np.isfinite(oof).all(axis=1)
        assert valid.sum()>10000 and len(np.unique(y_train[valid]))==2
        om=make_meta_features(oof[valid],X_train.iloc[np.flatnonzero(valid)])
        meta_models={}
        for name,cols in META_SETS.items():
            if name=='Stress-trained uncertainty–quality stacking':
                continue
            meta_models[name]=Pipeline([
                ('imp',SimpleImputer(strategy='median')),
                ('scale',StandardScaler()),
                ('meta',LogisticRegression(C=.2,max_iter=1500,random_state=SEED))])
            meta_models[name].fit(om[cols],y_train[valid])

        # Augmentasi OOF mencakup empat mekanisme, termasuk gangguan terkait kualitas data historis.
        # Setiap gangguan menghasilkan meta-fitur dari prediksi model yang dilatih di masa lalu.
        stress_x=[];stress_y=[]
        for foldnum,((fit,hold),fmodels) in enumerate(zip(folds,fold_predictors),start=1):
            begin=windows[foldnum-1][0];end=windows[foldnum-1][1]
            for j,(method,level) in enumerate(zip(STRESS_METHODS_OOF,STRESS_LEVELS_OOF)):
                print(f'OOF stress augmentation fold={foldnum}, mechanism={method}, rate={level:.0%}')
                block=fold_stress_meta(fmodels,hold,begin,end,method,level,
                    seed=SEED+int(foldnum*1000+j*100))
                stress_x.append(block)
                stress_y.append(y_train[hold])
        stress_m=pd.concat(stress_x,ignore_index=True)
        stress_y=np.concatenate(stress_y)
        aug_x=pd.concat([om,stress_m],ignore_index=True)
        aug_y=np.concatenate([y_train[valid],stress_y])
        clean_n=len(om);stress_n=len(stress_m)
        weights=np.concatenate([
            np.full(clean_n,STRESS_CLEAN_WEIGHT/clean_n),
            np.full(stress_n,(1-STRESS_CLEAN_WEIGHT)/stress_n)])
        aug_name='Stress-trained uncertainty–quality stacking'
        aug_cols=META_SETS[aug_name]
        stress_model=Pipeline([
            ('imp',SimpleImputer(strategy='median')),
            ('scale',StandardScaler()),
            ('meta',LogisticRegression(C=.2,max_iter=1500,random_state=SEED))])
        stress_model.fit(aug_x[aug_cols],aug_y,meta__sample_weight=weights*len(aug_y))
        meta_models[aug_name]=stress_model
        print('Stress meta observations:',len(aug_y),'| original OOF observations:',clean_n)

        base_models={name:clone(est).fit(X_train,y_train)
                     for name,est in make_base_models().items()}
        state={'model_signature':MODEL_SIGNATURE,
               'model_spec_version':MODEL_SPEC_VERSION,
               'source_hashes':SOURCE_SIGNATURE,
               'base_models':base_models,'meta_models':meta_models,
               'oof_valid_rows':int(valid.sum()),'oof_positive':int(y_train[valid].sum()),
               'stress_augmentation_rows':len(stress_m),
               'stress_mechanisms':STRESS_METHODS_OOF,
               'stress_levels':STRESS_LEVELS_OOF,
               'stress_clean_weight':STRESS_CLEAN_WEIGHT}
        tmp_cache=MODEL_CACHE.with_suffix('.joblib.tmp')
        joblib.dump(state,tmp_cache,compress=3)
        tmp_cache.replace(MODEL_CACHE)
        print('Model cache saved outside results directory:',MODEL_CACHE)
    base_models=state['base_models'];meta_models=state['meta_models']

    def raw_predictions(X):
        p=np.column_stack([base_models[n].predict_proba(X)[:,1] for n in BASE_NAMES])
        meta=make_meta_features(p,X)
        out={n:p[:,j] for j,n in enumerate(BASE_NAMES)}
        for n in META_NAMES:
            out[n]=meta_models[n].predict_proba(meta[META_SETS[n]])[:,1]
        return out

    # Baseline tanpa informasi hujan: hanya musim dan identitas wilayah.
    CLIM_FEATURES=['month_sin','month_cos'] + [c for c in FEATURE_COLUMNS if c.startswith('region_')]
    climatology=Pipeline([('scale',StandardScaler()),
        ('clf',LogisticRegression(C=.2,max_iter=1500,random_state=SEED))])
    climatology.fit(X_train[CLIM_FEATURES],y_train)
    _old_raw_predictions=raw_predictions
    def raw_predictions(X):
        out=_old_raw_predictions(X)
        out['Seasonal-region baseline']=climatology.predict_proba(X[CLIM_FEATURES])[:,1]
        return out
    val_raw=raw_predictions(X_val)
    test_raw=raw_predictions(X_test)
    future_raw=raw_predictions(X_future) if len(X_future) else {}
    print('Fitted model families:',len(ALL_NAMES),'| OOF rows:',state['oof_valid_rows'])
    print('Model fitting cutoff: 2023-12-31 | calibration cutoff: 2024-12-31 | 2025 status: exploratory')
    print('Model signature:', MODEL_SIGNATURE, '| cache:', MODEL_CACHE)

### B14. Validation-only calibration and model metrics

In [ ]:
if TRAIN_REQUIRED:
    # Kalibrasi menggunakan logits agar probabilitas di test tidak dipakai saat fitting.
    def logit(p):
        p=np.clip(np.asarray(p,dtype=float),1e-6,1-1e-6)
        return np.log(p/(1-p)).reshape(-1,1)
    calibrators={}
    thresholds={}
    for name in ALL_NAMES:
        calibrated=LogisticRegression(C=1.0,max_iter=1200,random_state=SEED)
        calibrated.fit(logit(val_raw[name]),y_val)
        calibrators[name]=calibrated
        pval=calibrated.predict_proba(logit(val_raw[name]))[:,1]
        # Maksimalkan F1 pada partisi validasi; test tidak pernah digunakan.
        p_curve,r_curve,candidate=precision_recall_curve(y_val,pval)
        f1_curve=2*p_curve[:-1]*r_curve[:-1]/np.maximum(1e-12,p_curve[:-1]+r_curve[:-1])
        thresholds[name]=float(candidate[int(np.argmax(f1_curve))])
        # Ambang dipilih pada data yang juga dipakai kalibrasi; metrik validasi bersifat apparent. 

    def calibrate(name,raw):
        return calibrators[name].predict_proba(logit(raw))[:,1]

    val_prob={n:calibrate(n,val_raw[n]) for n in ALL_NAMES}
    test_prob={n:calibrate(n,test_raw[n]) for n in ALL_NAMES}
    future_prob={n:calibrate(n,future_raw[n]) for n in ALL_NAMES} if future_raw else {}

    def ece_score(y,p,bins=10):
        y=np.asarray(y,dtype=int);p=np.asarray(p,dtype=float)
        edges=np.linspace(0,1,bins+1)
        cat=np.minimum(np.digitize(p,edges[1:-1]),bins-1)
        result=0.
        for i in range(bins):
            idx=cat==i
            if idx.any():result+=idx.mean()*abs(y[idx].mean()-p[idx].mean())
        return float(result)

    def calc_metrics(y,p,threshold):
        p=np.asarray(p,dtype=float)
        pred=p>=threshold
        return {'n':len(y),'positives':int(np.sum(y)),'positive_prevalence':float(np.mean(y)),
            'average_precision':float(average_precision_score(y,p)),
            'roc_auc':float(roc_auc_score(y,p)) if np.unique(y).size==2 else np.nan,
            'brier_score':float(brier_score_loss(y,p)),
            'log_loss':float(log_loss(y,np.clip(p,1e-6,1-1e-6),labels=[0,1])),
            'ece_10bins':ece_score(y,p,10),
            'brier_skill_vs_prevalence':float(1-brier_score_loss(y,p)/max(1e-12,float(np.mean(y)*(1-np.mean(y))))),
            'calibration_intercept':float(LogisticRegression(C=1e6,max_iter=1000).fit(logit(p),y).intercept_[0]) if len(np.unique(y))==2 else np.nan,
            'calibration_slope':float(LogisticRegression(C=1e6,max_iter=1000).fit(logit(p),y).coef_[0,0]) if len(np.unique(y))==2 else np.nan,
            'precision':float(precision_score(y,pred,zero_division=0)),
            'recall':float(recall_score(y,pred,zero_division=0)),
            'f1':float(f1_score(y,pred,zero_division=0)),
            'threshold_2024':float(threshold),
            'true_positives':int(np.sum((p>=threshold)&(np.asarray(y)==1))),
            'false_positives':int(np.sum((p>=threshold)&(np.asarray(y)==0))),
            'false_negatives':int(np.sum((p<threshold)&(np.asarray(y)==1))),
            'true_negatives':int(np.sum((p<threshold)&(np.asarray(y)==0)))}

    val_metrics=pd.DataFrame([{'model':name,'period':'2024_validation',
                 **calc_metrics(y_val,val_prob[name],thresholds[name])} for name in ALL_NAMES])
    test_metrics=pd.DataFrame([{'model':name,'period':'2025_exploratory_test',
                 **calc_metrics(y_test,test_prob[name],thresholds[name])} for name in ALL_NAMES])
    future_metrics=pd.DataFrame()
    if future_prob:
        future_rows=[]
        for name in ALL_NAMES:
            row={'model':name,'period':'prospective_2026plus',
                 **calc_metrics(y_future,future_prob[name],thresholds[name])}
            if np.unique(y_future).size<2:
                row['average_precision']=np.nan
                row['roc_auc']=np.nan
            row['evaluation_status']='future_period_not_used_for_fit_or_calibration; independence_attestation_required'
            future_rows.append(row)
        future_metrics=pd.DataFrame(future_rows)
    print('2025 retrospective comparison (exploratory, previously inspected):')
    display(test_metrics[['model','average_precision','brier_score','recall','f1','ece_10bins']]
            .sort_values('average_precision',ascending=False))
    if not future_metrics.empty:
        print('Post-2025 temporal holdout candidate (not used for fitting or calibration):')
        display(future_metrics[['model','n','positives','positive_prevalence','average_precision','brier_score','ece_10bins']])

    # Simpan artefak model di cache terpisah, bukan di folder hasil publikasi.
    # Bundle ini mengunci model, kalibrator, fitur, ambang, dan provenance untuk penggunaan ulang.
    MODEL_BUNDLE_PATH=CACHE/f'frozen_model_bundle_{MODEL_SIGNATURE}.joblib'
    frozen_bundle={
        'pipeline_version':PIPELINE_VERSION,
        'model_signature':MODEL_SIGNATURE,
        'experiment_signature':SIGNATURE,
        'source_doi':'10.17632/d7j3s64vvr.1',
        'source_hashes':SOURCE_SIGNATURE,
        'source_override_dir':str(SOURCE_OVERRIDE_DIR) if SOURCE_OVERRIDE_DIR else '',
        'target_definition':'recorded flood report on a region-day; no report is not verified physical absence',
        'training_period':'2020-01-01 through 2023-12-31',
        'calibration_and_threshold_period':'2024-01-01 through 2024-12-31',
        'exploratory_period':'2025-01-01 through 2025-12-31',
        'prospective_period':'2026-01-01 onward, if supplied',
        'feature_columns':FEATURE_COLUMNS,
        'feature_numeric':FEATURE_NUMERIC,
        'quality_columns':QUALITY_COLUMNS,
        'climatology_features':CLIM_FEATURES,
        'base_names':BASE_NAMES,
        'all_names':ALL_NAMES,
        'meta_sets':META_SETS,
        'base_models':base_models,
        'meta_models':meta_models,
        'seasonal_region_baseline':climatology,
        'calibrators':calibrators,
        'thresholds':thresholds,
        'training_diagnostics':state,
        'prediction_issue_time':'not verified; do not claim an operational lead time',
        'saved_at_utc':datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00','Z')
    }
    joblib.dump(frozen_bundle,MODEL_BUNDLE_PATH,compress=3)
    print('Frozen model bundle saved outside publication outputs:',MODEL_BUNDLE_PATH)

### B16. Station-level missingness stress evaluation

In [ ]:
if TRAIN_REQUIRED:
    # Membatasi eksperimen skenario ke periode 2025 dan memberi konteks lag Desember 2024.
    robust_rows=[]
    robust_models=['Logistic Regression','Random Forest','HistGradientBoosting',
                   'Standard stacking','Full uncertainty–quality stacking',
                   'Stress-trained uncertainty–quality stacking']
    observed_2025=raw_matrix.loc['2025-01-01':'2025-12-31']
    # Bobot kualitas dihitung sekali, bukan berulang pada setiap simulasi.
    HISTORICAL_STATION_MISSINGNESS=raw_matrix.loc['2020-01-01':'2023-12-31'].isna().mean().reindex(
        observed_2025.columns).fillna(0).to_numpy(dtype=float)

    def raw_predictions_robustness(X):
        """Prediksi cepat: base learners dan tiga meta-learner utama saja."""
        base_p=np.column_stack([base_models[n].predict_proba(X)[:,1] for n in BASE_NAMES])
        meta=make_meta_features(base_p,X)
        out={n:base_p[:,j] for j,n in enumerate(BASE_NAMES)}
        for n in ['Standard stacking','Full uncertainty–quality stacking',
                  'Stress-trained uncertainty–quality stacking']:
            out[n]=meta_models[n].predict_proba(meta[META_SETS[n]])[:,1]
        return out

    def mask_station_observations(base,method,level,seed):
        gen=np.random.default_rng(seed)
        x=base.copy()
        inyear=x.index.year==2025
        arr=x.loc[inyear].to_numpy(dtype=float,copy=True)
        observed=np.isfinite(arr)
        original_n=int(observed.sum())
        if level==0:return x,0.0
        to_mask=np.zeros_like(observed,dtype=bool)
        if method=='MCAR':
            to_mask=gen.random(arr.shape)<level
        elif method=='Station outage':
            nstation=max(1,round(arr.shape[1]*level))
            chosen=gen.choice(arr.shape[1],size=nstation,replace=False)
            to_mask[:,chosen]=True
        elif method=='Quality-dependent':
            # Peluang hilang lebih besar pada stasiun dengan missingness historis tinggi.
            hist_missing=HISTORICAL_STATION_MISSINGNESS
            weight=.1+hist_missing
            weight=weight/max(weight.mean(),1e-12)
            prob=np.minimum(level*weight,1.0)
            to_mask=gen.random(arr.shape)<prob[None,:]
        elif method=='Temporal block':
            # Setiap blok 7 hari per stasiun; target penghapusan didasarkan pada sel observasi.
            target=round(level*original_n)
            # Menghasilkan sekumpulan blok acak sekaligus untuk mempercepat Colab.
            # Blok dapat tumpang tindih; angka achieved missingness dicatat sesudah masking.
            batch=max(60,min(1000,int(target/max(1,BLOCK_LENGTH_DAYS*15))))
            for _ in range(250):
                if int(np.count_nonzero(to_mask & observed))>=target:break
                start=gen.integers(0,max(1,arr.shape[0]-BLOCK_LENGTH_DAYS+1),size=batch)
                col=gen.integers(0,arr.shape[1],size=batch)
                for lag in range(BLOCK_LENGTH_DAYS):
                    to_mask[np.minimum(start+lag,arr.shape[0]-1),col]=True
            if int(np.count_nonzero(to_mask & observed))<target:
                raise RuntimeError('Target kehilangan blok temporal tidak tercapai; periksa parameter.')
        else:raise ValueError('Mekanisme kehilangan data tidak diketahui')
        to_mask &= observed
        erased=int(to_mask.sum())
        arr[to_mask]=np.nan
        x.loc[inyear,:]=arr
        return x,erased/max(original_n,1)

    # Untuk uji cepat cukup satu tingkat dan satu seed; bukan hasil pelaporan.
    experiments=[('Original',0.0,0)]
    for method in ['MCAR','Station outage','Temporal block','Quality-dependent']:
        for level in ([.10] if FAST_TEST else MISSING_LEVELS[1:]):
            for rep in range(1 if FAST_TEST else ROBUSTNESS_REPS):
                experiments.append((method,level,rep))

    for ix,(method,level,rep) in enumerate(experiments,start=1):
        if method=='Original':
            px=X_test.copy();added_missing=0.
        else:
            # Konteks 2024-12-15 menjamin seluruh lag 7 hari tersedia pada awal 2025.
            partial=raw_matrix.loc['2024-12-15':'2025-12-31'].copy()
            masked,added_missing=mask_station_observations(
                partial,method,level,SEED+1000*int(level*100)+11*rep+{'MCAR':1,'Station outage':2,'Temporal block':3,'Quality-dependent':4}[method])
            f=make_features(masked)
            f=f[f.date.dt.year==2025].sort_values(['date','region_key']).reset_index(drop=True)
            reference=panel.loc[TEST_MASK,['date','region_key']].reset_index(drop=True)
            assert f[['date','region_key']].equals(reference),'Urutan tanggal/wilayah skenario tidak cocok.'
            px=X_test.copy()
            px[FEATURE_NUMERIC]=px[FEATURE_NUMERIC].astype(float)
            for col in FEATURE_NUMERIC:px[col]=f[col].to_numpy(dtype=float)
        raw=raw_predictions_robustness(px)
        for name in robust_models:
            probability=calibrate(name,raw[name])
            robust_rows.append({'mechanism':method,'target_additional_missing_fraction':level,
               'achieved_additional_missing_fraction':added_missing,'replicate':rep,
               'model':name,'average_precision':average_precision_score(y_test,probability),
               'brier_score':brier_score_loss(y_test,probability),
               'ece_10bins':ece_score(y_test,probability,10)})
        # Baseline musim-wilayah tidak menggunakan hujan, sehingga hasilnya identik untuk semua masking.
        baseline_probability=test_prob['Seasonal-region baseline']
        robust_rows.append({'mechanism':method,'target_additional_missing_fraction':level,
           'achieved_additional_missing_fraction':added_missing,'replicate':rep,
           'model':'Seasonal-region baseline',
           'average_precision':average_precision_score(y_test,baseline_probability),
           'brier_score':brier_score_loss(y_test,baseline_probability),
           'ece_10bins':ece_score(y_test,baseline_probability,10)})
        if ix==1 or ix==len(experiments) or ix%5==0:
            print(f'Missingness experiments {ix}/{len(experiments)} completed')
    robustness=pd.DataFrame(robust_rows)

### B18. Paired temporal bootstrap

In [ ]:
if TRAIN_REQUIRED:
    # Mempertahankan pasangan prediksi model pada tanggal yang sama dalam setiap bootstrap.
    test_dates=panel.loc[TEST_MASK,'date'].to_numpy()
    unique_dates=np.sort(np.unique(test_dates))
    index_by_day={pd.Timestamp(d):np.flatnonzero(test_dates==d) for d in unique_dates}
    N_DAYS=len(unique_dates)

    # Perbandingan utama dan ablasi memakai prediksi test yang sama.
    comparisons=[('Full vs standard','Full uncertainty–quality stacking','Standard stacking'),
                 ('Uncertainty vs standard','Uncertainty-only stacking','Standard stacking'),
                 ('Quality vs standard','Quality-only stacking','Standard stacking'),
                 ('Full vs uncertainty','Full uncertainty–quality stacking','Uncertainty-only stacking'),
                 ('Full vs HistGradientBoosting','Full uncertainty–quality stacking','HistGradientBoosting'),
                 ('Full vs seasonal-region baseline','Full uncertainty–quality stacking','Seasonal-region baseline'),
                 ('Stress-trained vs standard','Stress-trained uncertainty–quality stacking','Standard stacking'),
                 ('Stress-trained vs full','Stress-trained uncertainty–quality stacking','Full uncertainty–quality stacking'),
                 ('Stress-trained vs HistGradientBoosting','Stress-trained uncertainty–quality stacking','HistGradientBoosting')]
    bootstrap_rows=[]
    b_rng=np.random.default_rng(SEED+999)
    R=60 if FAST_TEST else BOOTSTRAP_REPS
    for label,a,b in comparisons:
        dif_ap=[];dif_bs=[]
        for rep in range(R):
            starts=b_rng.integers(0,N_DAYS,size=int(np.ceil(N_DAYS/BLOCK_LENGTH_DAYS)))
            sampled_days=sample_non_circular_blocks(N_DAYS, BLOCK_LENGTH_DAYS, b_rng)
            indices=np.concatenate([index_by_day[pd.Timestamp(unique_dates[j])] for j in sampled_days])
            yy=y_test[indices]
            if yy.min()==yy.max():continue
            pa=test_prob[a][indices];pb=test_prob[b][indices]
            dif_ap.append(average_precision_score(yy,pa)-average_precision_score(yy,pb))
            dif_bs.append(brier_score_loss(yy,pb)-brier_score_loss(yy,pa))
        observed_ap=average_precision_score(y_test,test_prob[a])-average_precision_score(y_test,test_prob[b])
        observed_bs=brier_score_loss(y_test,test_prob[b])-brier_score_loss(y_test,test_prob[a])
        bootstrap_rows.append({'comparison':label,'model_a':a,'model_b':b,
            'test_ap_difference_a_minus_b':observed_ap,
            'ap_delta_ci_low':float(np.quantile(dif_ap,.025)),
            'ap_delta_ci_high':float(np.quantile(dif_ap,.975)),
            'test_brier_improvement_b_minus_a':observed_bs,
            'brier_delta_ci_low':float(np.quantile(dif_bs,.025)),
            'brier_delta_ci_high':float(np.quantile(dif_bs,.975)),
            'bootstrap_reps_successful':len(dif_ap),
            'resampling_unit':f'{BLOCK_LENGTH_DAYS}-day date block, all 27 regions'})
    # Sensitivitas asumsi autokorelasi: blok 14 dan 30 hari, selain analisis utama 7 hari.
    # Analisis tambahan tetap berpasangan berdasarkan tanggal dan menjaga 27 wilayah bersama.
    for sensitivity_days in [14,30]:
        r_sens=30 if FAST_TEST else 300
        sens_rng=np.random.default_rng(SEED+9000+sensitivity_days)
        for label, aa, bb in [
            ('Full vs standard','Full uncertainty–quality stacking','Standard stacking'),
            ('Full vs HistGradientBoosting','Full uncertainty–quality stacking','HistGradientBoosting'),
            ('Stress-trained vs standard','Stress-trained uncertainty–quality stacking','Standard stacking')]:
            aps=[]; bs=[]
            for rep in range(r_sens):
                sampled=sample_non_circular_blocks(N_DAYS, sensitivity_days, sens_rng)
                idx=np.concatenate([index_by_day[pd.Timestamp(unique_dates[j])] for j in sampled])
                yy=y_test[idx]
                if yy.min()==yy.max(): continue
                pa=test_prob[aa][idx];pb=test_prob[bb][idx]
                aps.append(average_precision_score(yy,pa)-average_precision_score(yy,pb))
                bs.append(brier_score_loss(yy,pb)-brier_score_loss(yy,pa))
            bootstrap_rows.append({
                'comparison':f'{label} [{sensitivity_days}-day sensitivity]',
                'model_a':aa,'model_b':bb,
                'test_ap_difference_a_minus_b':average_precision_score(y_test,test_prob[aa])-average_precision_score(y_test,test_prob[bb]),
                'ap_delta_ci_low':float(np.quantile(aps,.025)),
                'ap_delta_ci_high':float(np.quantile(aps,.975)),
                'test_brier_improvement_b_minus_a':brier_score_loss(y_test,test_prob[bb])-brier_score_loss(y_test,test_prob[aa]),
                'brier_delta_ci_low':float(np.quantile(bs,.025)),
                'brier_delta_ci_high':float(np.quantile(bs,.975)),
                'bootstrap_reps_successful':len(aps),
                'resampling_unit':f'{sensitivity_days}-day date block, all 27 regions'})
    bootstrap_comparison=pd.DataFrame(bootstrap_rows)
    print('Paired test comparisons (positive Brier improvement = better model A):')
    display(bootstrap_comparison[['comparison','test_ap_difference_a_minus_b',
        'ap_delta_ci_low','ap_delta_ci_high','test_brier_improvement_b_minus_a']])

### B20. Full evidence CSV export and audit

In [ ]:
if TRAIN_REQUIRED:
    # Merangkum informasi kualitas data berdasarkan wilayah dan tahun.
    regional_quality=(panel.groupby(['region_key','year'],as_index=False)
        .agg(region_days=('date','size'),positive_region_days=('flood_recorded','sum'),
             flood_reports=('reported_flood_events','sum'),
             mean_available_station_fraction_lag1=('coverage_lag1','mean'),
             mean_station_count=('expected_station_count','mean')))
    regional_quality['recorded_event_rate']=regional_quality.positive_region_days/regional_quality.region_days
    # Evaluasi heterogenitas wilayah 2025; AP tidak didefinisikan bila tidak ada positif.
    regional_pred=panel.loc[TEST_MASK,['region_key','flood_recorded','coverage_lag1']].reset_index(drop=True)
    regional_pred['p_full']=test_prob['Full uncertainty–quality stacking']
    regional_pred['p_hist']=test_prob['HistGradientBoosting']
    regional_pred['p_standard']=test_prob['Standard stacking']
    regional_pred['p_stress']=test_prob['Stress-trained uncertainty–quality stacking']
    subgroup=[]
    for region,group in regional_pred.groupby('region_key'):
        yy=group.flood_recorded.to_numpy()
        subgroup.append({'region_key':region,'year':2025,
          'test_ap_full':average_precision_score(yy,group.p_full) if yy.sum()>0 else np.nan,
          'test_ap_histgradientboosting':average_precision_score(yy,group.p_hist) if yy.sum()>0 else np.nan,
          'test_brier_full':brier_score_loss(yy,group.p_full),
          'test_brier_histgradientboosting':brier_score_loss(yy,group.p_hist),
          'test_brier_standard':brier_score_loss(yy,group.p_standard),
          'test_brier_stress':brier_score_loss(yy,group.p_stress),
          'test_ap_stress':average_precision_score(yy,group.p_stress) if yy.sum()>0 else np.nan,
          'test_ap_standard':average_precision_score(yy,group.p_standard) if yy.sum()>0 else np.nan,
          'region_test_positive_n':int(yy.sum()),
          'region_test_mean_station_coverage':float(group.coverage_lag1.mean()),
          'region_test_ap_reliability_flag':('limited_positive_events' if yy.sum()<10 else 'descriptive_only')})
    regional_quality=regional_quality.merge(pd.DataFrame(subgroup),on=['region_key','year'],how='left')
    regional_quality['doi']='10.17632/d7j3s64vvr.1'
    regional_quality['evaluation_status']='exploratory_test_2025_inspected_in_previous_versions'
    regional_quality['label_note']='No record does not confirm absence of a flood'
    regional_quality['pipeline_version']=PIPELINE_VERSION
    regional_quality['source_signature']=SIGNATURE

    split_balance=(panel.groupby(['data_partition','year'],as_index=False)
        .agg(region_days=('flood_recorded','size'),recorded_flood_region_days=('flood_recorded','sum'),
             flood_report_count=('reported_flood_events','sum'),
             mean_lag1_coverage=('coverage_lag1','mean')))
    split_balance['recorded_event_prevalence']=split_balance.recorded_flood_region_days/split_balance.region_days
    split_balance['observations_in_source_rainfall_matrix']=len(rain_raw)
    split_balance['mapped_rainfall_stations']=len(kept_cols)
    split_balance['unmapped_rainfall_stations']=len(unresolved)
    split_balance['source_report_count']=len(flood_raw)

    # Simpan 2025 prediksi semua model untuk reproduksi tabel dan gambar analisis.
    test_metrics['evaluation_status']='exploratory_reused_2025_test'
    val_metrics['evaluation_status']='in_sample_calibration_and_threshold_selection'
    bootstrap_comparison['evaluation_status']='exploratory_reused_2025_test'
    robustness['evaluation_status']='exploratory_reused_2025_test'
    robustness['replicate_interval_note']='Percentiles across simulated perturbations are simulation spread, not 95% statistical confidence intervals.'

    # Simpan semua prediksi test lengkap untuk audit independen.
    test_predictions=panel.loc[TEST_MASK,['date','region_key','flood_recorded',
            'reported_flood_events','coverage_lag1','coverage_prev7_mean']].reset_index(drop=True)
    for name in ALL_NAMES:
        short=re.sub('[^a-z0-9]+','_',name.lower()).strip('_')
        test_predictions['p_'+short]=test_prob[name]
        test_predictions['pred_'+short]=(test_prob[name]>=thresholds[name]).astype(int)
    test_predictions['model_signature']=MODEL_SIGNATURE
    test_predictions['target_definition']='recorded flood report on a region-day'

    # Jika workbook diperluas setelah 2025, prediksi periode tersebut memakai model dan kalibrator terkunci.
    # File ini disimpan di cache, tidak di folder publikasi yang harus berisi tepat 20 artefak.
    FUTURE_PREDICTIONS_PATH=CACHE/f'prospective_predictions_{MODEL_SIGNATURE}.csv'
    future_predictions=pd.DataFrame()
    if len(future_idx):
        future_predictions=panel.loc[PROSPECTIVE_MASK,['date','region_key','flood_recorded',
            'reported_flood_events','coverage_lag1','coverage_prev7_mean']].reset_index(drop=True)
        for name in ALL_NAMES:
            short=re.sub('[^a-z0-9]+','_',name.lower()).strip('_')
            future_predictions['p_'+short]=future_prob[name]
            future_predictions['pred_'+short]=(future_prob[name]>=thresholds[name]).astype(int)
        future_predictions['model_signature']=MODEL_SIGNATURE
        future_predictions['target_definition']='recorded flood report on a region-day'
        future_predictions['label_source']='source workbook flood reports; physical absence not verified'
        future_predictions['independence_attestation_required']=True
        future_predictions.to_csv(FUTURE_PREDICTIONS_PATH,index=False,encoding='utf-8-sig')
        print('Prospective predictions saved outside publication outputs:',FUTURE_PREDICTIONS_PATH)
    elif FUTURE_PREDICTIONS_PATH.exists():
        FUTURE_PREDICTIONS_PATH.unlink()

    # Reliability bins lengkap, termasuk bin kosong, untuk seluruh model.
    rel_rows=[]
    for name in ALL_NAMES:
        p=test_prob[name]
        idx=np.minimum(np.digitize(p,np.linspace(0,1,11)[1:-1]),9)
        for k in range(10):
            m=idx==k
            rel_rows.append({'model':name,'bin':k+1,'bin_lower':k/10,'bin_upper':(k+1)/10,
                'n':int(m.sum()),'mean_predicted':float(np.mean(p[m])) if m.any() else np.nan,
                'observed_fraction':float(np.mean(y_test[m])) if m.any() else np.nan})
    # Equal-frequency calibration bins lebih informatif saat prediksi terkonsentrasi mendekati nol.
    for name in ALL_NAMES:
        p=np.asarray(test_prob[name],dtype=float)
        rank=pd.Series(p).rank(method='first').to_numpy()
        quantile_id=pd.qcut(rank,10,labels=False,duplicates='drop')
        for k in range(10):
            m=np.asarray(quantile_id)==k
            rel_rows.append({'model':name,'bin':k+1,'bin_lower':float(np.min(p[m])) if m.any() else np.nan,
                'bin_upper':float(np.max(p[m])) if m.any() else np.nan,
                'n':int(m.sum()),'mean_predicted':float(np.mean(p[m])) if m.any() else np.nan,
                'observed_fraction':float(np.mean(y_test[m])) if m.any() else np.nan,
                'binning':'equal_frequency'})
    reliability_bins=pd.DataFrame(rel_rows)
    reliability_bins['binning']=reliability_bins['binning'].fillna('equal_width')

    # Ringkasan distribusi replikasi per mekanisme dan tingkat kehilangan.
    robustness_summary=(robustness.groupby(['mechanism','target_additional_missing_fraction','model'],as_index=False)
      .agg(ap_mean=('average_precision','mean'),ap_std=('average_precision','std'),
           ap_p025=('average_precision',lambda x:np.quantile(x,.025)),
           ap_p975=('average_precision',lambda x:np.quantile(x,.975)),
           achieved_mean=('achieved_additional_missing_fraction','mean')))
    # Lampirkan rerata dan rentang empiris ke baris tingkat replikasi, bukan sebagai CI formal populasi.
    robustness=robustness.merge(robustness_summary,
      on=['mechanism','target_additional_missing_fraction','model'],how='left')


    # AUDIT V7: model v6 tidak disetel ulang menggunakan test 2025.
    # Memeriksa berapa banyak kejadian positif ditemukan dalam kuota alarm 1%, 3%, dan 5%.
    # Kuota ini bukan kebijakan operasional yang telah divalidasi secara prospektif.
    def alert_budget_stats(y, probabilities, fraction):
        y=np.asarray(y,dtype=int)
        p=np.asarray(probabilities,dtype=float)
        assert y.size==p.size and np.isfinite(p).all()
        budget=max(1,int(np.ceil(fraction*len(y))))
        top=np.argsort(-p,kind='stable')[:budget]
        positives_caught=int(y[top].sum())
        precision=positives_caught/budget
        recall=positives_caught/max(1,int(y.sum()))
        baseline=max(float(y.mean()),1e-12)
        return {'budget_n':budget,'hits_n':positives_caught,
                'precision':float(precision),'recall':float(recall),
                'lift_over_prevalence':float(precision/baseline)}

    for metric_frame,probabilities,labels in [(test_metrics,test_prob,y_test),
                                               (val_metrics,val_prob,y_val)]:
        for idx,row in metric_frame.iterrows():
            model=row['model']
            for frac in [0.01,0.03,0.05]:
                key=f'top_{int(frac*100)}pct'
                vals=alert_budget_stats(labels,probabilities[model],frac)
                for field,value in vals.items():
                    metric_frame.loc[idx,key+'_'+field]=value
        metric_frame['alert_budget_status']='descriptive_not_deployment_validated'
        metric_frame['evidence_audit_version']=EVIDENCE_AUDIT_VERSION

    # AUDIT V7: untuk setiap masking identik hitung selisih secara berpasangan.
    # Hasil menang/kalah antar simulasi bukan uji signifikansi independen.
    pair_keys=['mechanism','target_additional_missing_fraction','replicate']
    for comp,suffix in [
        ('Standard stacking','standard'),
        ('HistGradientBoosting','histgradientboosting'),
        ('Full uncertainty–quality stacking','full'),
        ('Stress-trained uncertainty–quality stacking','stress')]:
        ref=robustness.loc[robustness.model.eq(comp),pair_keys+['average_precision','brier_score']].copy()
        assert not ref.duplicated(pair_keys).any(), 'Satu model muncul lebih dari sekali dalam pasangan masking.'
        ref=ref.rename(columns={
            'average_precision':f'reference_ap_{suffix}',
            'brier_score':f'reference_brier_{suffix}'})
        robustness=robustness.merge(ref,on=pair_keys,how='left',validate='many_to_one')
        robustness[f'ap_minus_{suffix}']=robustness.average_precision-robustness[f'reference_ap_{suffix}']
        robustness[f'brier_improvement_over_{suffix}']=robustness[f'reference_brier_{suffix}']-robustness.brier_score
        assert robustness[f'reference_ap_{suffix}'].notna().all(), f'Pembanding {comp} hilang.'

    # Ringkasan pasangan per skenario diulang pada setiap baris model agar tetap 10 CSV.
    for suffix in ['standard','histgradientboosting','full','stress']:
        group_cols=['mechanism','target_additional_missing_fraction','model']
        grouped=robustness.groupby(group_cols,dropna=False)
        rob_summary=grouped.agg(
            **{f'mean_delta_ap_vs_{suffix}':(f'ap_minus_{suffix}','mean'),
               f'fraction_positive_ap_vs_{suffix}':(f'ap_minus_{suffix}',lambda v:float((v>0).mean())),
               f'mean_brier_improvement_vs_{suffix}':(f'brier_improvement_over_{suffix}','mean'),
               f'fraction_better_brier_vs_{suffix}':(f'brier_improvement_over_{suffix}',lambda v:float((v>0).mean()))}
        ).reset_index()
        robustness=robustness.merge(rob_summary,on=group_cols,how='left',validate='many_to_one')
    robustness['simulation_spread_not_confidence_interval']=True
    robustness['evidence_audit_version']=EVIDENCE_AUDIT_VERSION

    # Audit area dengan jaringan tanpa pengamatan atau cakupan di bawah 50%.
    coverage_panel=panel.loc[panel.date.gt(panel.date.min())].copy()
    coverage_audit=(coverage_panel.groupby(['region_key','year'],as_index=False)
        .agg(lag1_no_station_days=('observed_stations_lag1',lambda v:int(v.fillna(0).eq(0).sum())),
             lag1_coverage_below_half_days=('coverage_lag1',lambda v:int(v.fillna(0).lt(.5).sum())),
             lag1_missing_coverage_days=('coverage_lag1',lambda v:int(v.isna().sum()))))
    regional_quality=regional_quality.merge(coverage_audit,on=['region_key','year'],how='left',validate='one_to_one')
    regional_quality['first_source_day_excluded_from_lag_coverage_audit']=True
    regional_quality['evidence_audit_version']=EVIDENCE_AUDIT_VERSION
    regional_quality['generalization_warning']='2025_inspected_repeatedly_exploratory_only'

    station_audit['station_id_confirmed']=station_audit['station_id'].notna() & station_audit['station_id'].astype(str).str.strip().ne('')
    station_audit['evidence_audit_version']=EVIDENCE_AUDIT_VERSION
    split_balance['evidence_audit_version']=EVIDENCE_AUDIT_VERSION
    bootstrap_comparison['evidence_audit_version']=EVIDENCE_AUDIT_VERSION
    bootstrap_comparison['multiple_comparisons_note']='CIs are unadjusted for multiple model comparisons.'
    bootstrap_comparison['confirmatory_claim_permitted']=False
    for i,rr in bootstrap_comparison.iterrows():
        bootstrap_comparison.loc[i,'ap_ci_excludes_zero']=bool(rr.ap_delta_ci_low>0 or rr.ap_delta_ci_high<0)
        bootstrap_comparison.loc[i,'brier_ci_excludes_zero']=bool(rr.brier_delta_ci_low>0 or rr.brier_delta_ci_high<0)

    # Diagnostik yang wajib tercetak sebelum mengekspor figur/tabel.
    checks=robustness[(robustness.mechanism!='Original') &
                      robustness.model.eq('Stress-trained uncertainty–quality stacking')]
    print('V7 paired simulated-missingness audit:',len(checks),'matched stress-trained replicates')
    print('  Stress-trained AP wins vs standard:',int((checks.ap_minus_standard>0).sum()),'/',len(checks))
    print('  Stress-trained Brier wins vs standard:',int((checks.brier_improvement_over_standard>0).sum()),'/',len(checks))
    print('  Mean AP difference vs standard:',round(float(checks.ap_minus_standard.mean()),6))
    print('  Mean Brier improvement vs standard:',round(float(checks.brier_improvement_over_standard.mean()),8))
    print('Regions/years with no observed station-days:',
          regional_quality.loc[regional_quality.lag1_no_station_days>0,
                               ['region_key','year','lag1_no_station_days']].to_string(index=False))

    # V8: menambahkan audit kalibrasi, kualitas wilayah, dan kuota alarm tanpa melatih ulang.
    # Penilaian ini bersifat eksploratif karena data 2025 sudah dianalisis pada iterasi sebelumnya.
    from math import sqrt

    def wilson_interval(k, n, z=1.959963984540054):
        # CI binomial marginal; tidak memperhitungkan korelasi antartanggal/wilayah.
        if n <= 0: return np.nan, np.nan
        p=k/n; d=1+z*z/n
        middle=(p+z*z/(2*n))/d
        spread=z*sqrt((p*(1-p)+z*z/(4*n))/n)/d
        return max(0.,middle-spread),min(1.,middle+spread)

    # Laporkan hitungan positif dan CI untuk tiap bin, tanpa menggunakan hasil untuk penyetelan model.
    reliability_bins['observed_positive_n'] = np.nan
    reliability_bins['observed_fraction_wilson_low'] = np.nan
    reliability_bins['observed_fraction_wilson_high'] = np.nan
    for i,row in reliability_bins.iterrows():
        if int(row['n']) == 0: continue
        k=int(round(float(row['observed_fraction'])*int(row['n'])))
        lo,hi=wilson_interval(k,int(row['n']))
        reliability_bins.loc[i,'observed_positive_n']=k
        reliability_bins.loc[i,'observed_fraction_wilson_low']=lo
        reliability_bins.loc[i,'observed_fraction_wilson_high']=hi
    reliability_bins['wilson_assumption']='marginal_binomial_approximation_not_cluster_adjusted'

    # Brier skill dinilai juga terhadap baseline musim-wilayah, bukan hanya prevalensi konstan.
    for metrics in (val_metrics,test_metrics):
        reference=float(metrics.loc[metrics.model.eq('Seasonal-region baseline'),'brier_score'].iloc[0])
        metrics['brier_skill_vs_seasonal_region']=1-metrics.brier_score/reference
        metrics['test_exposure_warning']='2025_reused_during_v3_to_v7_research_development'
        metrics['multiple_comparisons_warning']='exploratory_unadjusted_comparisons'

    # Ringkasan kuota alarm terstruktur; tidak dijadikan alat memilih threshold.
    operational_rows=[]
    for model in ALL_NAMES:
        for fraction in (.01,.03,.05,.10,.20):
            row=alert_budget_stats(y_test,test_prob[model],fraction)
            operational_rows.append({
                'model':model,'test_year':2025,'alert_budget_fraction':fraction,
                'alert_budget_n':row['budget_n'],'recorded_events_caught':row['hits_n'],
                'false_alerts_n':row['budget_n']-row['hits_n'],
                'precision':row['precision'],'recall':row['recall'],
                'lift_over_prevalence':row['lift_over_prevalence'],
                'test_region_days':len(y_test),'recorded_positive_region_days':int(y_test.sum()),
                'evaluation_status':'descriptive_not_operationally_validated'
            })
    operational_budget=pd.DataFrame(operational_rows)

    # Penilaian subkelompok didasarkan pada cakupan jaringan pada hari sebelum target.
    # Titik batas ditetapkan untuk audit deskriptif dan TIDAK untuk optimasi model.
    coverage_for_test=np.asarray(panel.loc[TEST_MASK,'coverage_lag1'],dtype=float)
    coverage_levels=[('0–49% coverage',0.,.5),('50–89% coverage',.5,.9),
                     ('90–98% coverage',.9,.99),('99–100% coverage',.99,1.000001)]
    coverage_rows=[]
    for name,lower,upper in coverage_levels:
        subset=np.isfinite(coverage_for_test) & (coverage_for_test>=lower) & (coverage_for_test<upper)
        n=int(subset.sum()); pos=int(y_test[subset].sum())
        for model in ['Seasonal-region baseline','HistGradientBoosting','Standard stacking',
                      'Full uncertainty–quality stacking','Stress-trained uncertainty–quality stacking']:
            if n == 0: continue
            proba=np.asarray(test_prob[model])[subset]
            coverage_rows.append({
                'coverage_group':name,'coverage_lower':lower,'coverage_upper':upper,
                'n':n,'positives':pos,'positive_prevalence':pos/n,'model':model,
                'mean_observed_coverage':float(np.mean(coverage_for_test[subset])),
                'average_precision_if_supported':(
                    float(average_precision_score(y_test[subset],proba)) if pos>=10 and n-pos>=10 else np.nan
                ),
                'brier_score':float(brier_score_loss(y_test[subset],proba)),
                'ap_support_flag':'supported_descriptive' if pos>=10 and n-pos>=10 else 'insufficient_positive_support',
                'evaluation_status':'exploratory_quality_stratification'
            })
    coverage_strata=pd.DataFrame(coverage_rows)

    # Koreksi audit wilayah: jangan laporkan AP wilayah yang hanya berisi 1–9 kejadian positif
    # sebagai angka yang seolah-olah memiliki reliabilitas yang cukup.
    for col in ['test_ap_full','test_ap_histgradientboosting','test_ap_stress','test_ap_standard']:
        mask=regional_quality.region_test_positive_n.notna() & (regional_quality.region_test_positive_n<10)
        regional_quality.loc[mask,col]=np.nan
    regional_quality['subgroup_ap_min_positive_cases']=10
    regional_quality['zero_rainfall_stations_not_equivalent_to_dry_weather']=True

    # Audit konsistensi peringkat dan ketersediaan stasiun; gagal keras jika label berubah.
    assert len(operational_budget)==len(ALL_NAMES)*5
    assert int(operational_budget.loc[
        (operational_budget.model=='Stress-trained uncertainty–quality stacking') &
        (operational_budget.alert_budget_fraction==.05),'recorded_events_caught'].iloc[0])>=0
    assert coverage_strata.groupby('model')['n'].sum().min()<=len(y_test)
    assert int(panel.loc[panel.year.le(2025),'flood_recorded'].sum())==1358
    print('2025 AP, Brier, and alert capture are exploratory, not confirmatory.')
    if len(future_predictions):
        print('Future predictions are generated with models fitted only through 2023 and calibrated only on 2024.')
    print(operational_budget.loc[operational_budget.alert_budget_fraction.eq(.05),
          ['model','recorded_events_caught','false_alerts_n','precision','recall']].to_string(index=False))
    print('Coverage groups (test 2025):')
    print(coverage_strata.loc[coverage_strata.model.eq('Stress-trained uncertainty–quality stacking'),
          ['coverage_group','n','positives','mean_observed_coverage','ap_support_flag']].to_string(index=False))


    # Hapus file hasil lama sebelum ekspor agar tidak tercampur dengan eksperimen sebelumnya.
    for p in OUTPUT.iterdir():
        if p.is_file() and p.suffix.lower() in ['.png','.csv']:
            p.unlink()

    metric_parts=[val_metrics,test_metrics]
    if not future_metrics.empty:
        metric_parts.append(future_metrics)
    csv_dict={
    '01_station_mapping_audit.csv':station_audit,
    '02_region_year_quality.csv':regional_quality,
    '03_lagged_feature_panel.csv':panel,
    '04_temporal_split_balance.csv':split_balance,
    '05_model_metrics_validation_and_test.csv':pd.concat(metric_parts,ignore_index=True),
    '06_held_out_test_predictions.csv':test_predictions,
    '07_test_reliability_bins.csv':reliability_bins,
    '08_station_missingness_robustness.csv':robustness,
    '09_operational_alert_budgets.csv':operational_budget,
    '10_paired_block_bootstrap.csv':bootstrap_comparison}
    for name,frame in csv_dict.items():frame.to_csv(OUTPUT/name,index=False,encoding='utf-8-sig')
    assert len(list(OUTPUT.glob('*.csv')))==10
    print('Exported 10 CSV research tables into:',OUTPUT)

In [ ]:
# Uji ulang bukti training yang baru dibentuk sebelum tahap audit publikasi dimulai.
if TRAIN_REQUIRED:
    ACTIVE_EVIDENCE_FILES=find_evidence_set(OUTPUT)
    assert ACTIVE_EVIDENCE_FILES is not None,'Ten training-evidence CSVs were not exported.'
    valid,reason=evidence_fullrun_checks(ACTIVE_EVIDENCE_FILES,require_full=not FAST_TEST)
    if not valid: raise RuntimeError('Training evidence failed validation: '+reason)
    # Tambahkan prediksi temporal pasca-2025 sebagai sumber opsional, tanpa memasukkannya ke 10 CSV publikasi.
    future_path=CACHE/f'prospective_predictions_{MODEL_SIGNATURE}.csv'
    if future_path.exists():
        ACTIVE_EVIDENCE_FILES['future_pred']=future_path
print('Auditing the evidence in:',next(iter(ACTIVE_EVIDENCE_FILES.values())).parent)

## C. Independent frozen-prediction audit: provenance, leakage, event definitions

The following stage uses **saved, immutable observation-level model probabilities**. It does not re-fit classifiers, recalibrate on 2025, or change thresholds. Assertions deliberately stop execution when labels, dates, missingness simulations, or model probabilities are inconsistent. The *at-risk* onset sensitivity conditions on having **no recorded flood in the previous seven full days**; it remains a retrospective administrative proxy rather than validated hydrological onset.

In [ ]:
# Pisahkan output penelitian dari CSV intermediate agar ZIP final hanya memuat tepat 20 artefak.
import os, sys, subprocess, importlib.util, warnings, json, hashlib, zipfile
from pathlib import Path
from scipy.optimize import minimize
from scipy.stats import norm
from sklearn.metrics import average_precision_score,roc_auc_score,brier_score_loss,precision_recall_curve
from PIL import Image
from matplotlib.ticker import PercentFormatter
DOI='10.17632/d7j3s64vvr.1'
VERSION='v18.0-teknika-sinta2-execution-guardrails'
RANDOM_SEED=20261018
N_BOOT_PRIMARY=1000 if not FAST_TEST else 35
N_BOOT_SENSITIVITY=300 if not FAST_TEST else 15
N_CALIBRATION_BOOT=600 if not FAST_TEST else 35
BLOCK_LENGTHS=(7,14,30)
IS_COLAB=IN_COLAB
COLORS={'blue':'#0072B2','orange':'#E69F00','green':'#009E73','red':'#D55E00',
        'purple':'#CC79A7','sky':'#56B4E9','black':'#111111','gray':'#666666','yellow':'#F0E442'}
OUTPUT=ROOT/'v18_teknika_publication_evidence_outputs';OUTPUT.mkdir(parents=True,exist_ok=True)
for old in list(OUTPUT.glob('*.png'))+list(OUTPUT.glob('*.csv')):old.unlink()
raw_hash={key:hashlib.sha256(p.read_bytes()).hexdigest() for key,p in ACTIVE_EVIDENCE_FILES.items()}
source_signature=hashlib.sha256(''.join(raw_hash[k] for k in sorted(raw_hash)).encode()).hexdigest()
if STRICT_HISTORICAL_SHA and source_signature!=STRICT_HISTORICAL_SHA:
    raise AssertionError('Evidence SHA-256 differs from explicitly pinned historical run')
frames={key:pd.read_csv(p,encoding='utf-8-sig') for key,p in ACTIVE_EVIDENCE_FILES.items()}
print('Frozen full-evidence SHA-256:',source_signature)
print('Evidence summary:',{key:len(frame) for key,frame in frames.items()})

### C5. Exact input and leakage checks

In [ ]:
# Audit data historis 2020–2025 terpisah dari prediksi temporal 2026+ bila tersedia.
panel_all=frames['panel'].copy();panel_all['date']=pd.to_datetime(panel_all['date'],errors='raise')
panel=panel_all[panel_all['date'].dt.year.le(2025)].copy()
pred=frames['pred'].copy();pred['date']=pd.to_datetime(pred['date'],errors='raise')
future_pred=frames.get('future_pred',pd.DataFrame()).copy()
if not future_pred.empty:
    future_pred['date']=pd.to_datetime(future_pred['date'],errors='raise')
split=frames['split'].copy();quality=frames['quality'].copy()
assert len(panel)==59184 and int(panel['flood_recorded'].sum())==1358
assert len(pred)==9855 and int(pred['flood_recorded'].sum())==296
assert pred['region_key'].nunique()==27 and pred['date'].nunique()==365
assert panel.duplicated(['date','region_key']).sum()==0
assert pred.duplicated(['date','region_key']).sum()==0
assert set(pred.date.dt.year)=={2025}
assert panel['flood_recorded'].isin([0,1]).all(), 'Target label must be binary.'
expected_dates=pd.date_range('2020-01-01','2025-12-31',freq='D')
assert panel['date'].nunique()==len(expected_dates), 'Historical daily calendar has missing or extra dates.'
assert set(pd.to_datetime(panel['date'].unique()))==set(expected_dates), 'Historical calendar is not complete.'
assert panel.groupby('region_key')['date'].nunique().eq(len(expected_dates)).all(), 'At least one region lacks daily rows.'
assert panel.loc[panel.year.between(2020,2023),'data_partition'].eq('training').all()
# Terima dua label yang terdokumentasi dari cache lama, tetapi hanya untuk tahun yang tepat.
assert panel.loc[panel.year.eq(2024),'data_partition'].isin(['calibration_validation','validation']).all()
assert panel.loc[panel.year.eq(2025),'data_partition'].isin(['exploratory_test_2025','test']).all()
check=pred[['date','region_key','flood_recorded']].merge(
    panel[['date','region_key','flood_recorded']],on=['date','region_key'],validate='one_to_one',suffixes=('_pred','_panel'))
assert len(check)==len(pred) and (check.flood_recorded_pred==check.flood_recorded_panel).all()

# Semua fitur hujan harus berasal dari lag atau jendela waktu sebelumnya; kolom target dilarang sebagai fitur.
feature_columns=[c for c in panel.columns if c not in {
    'date','region_key','year','data_partition','reported_flood_events','flood_recorded','outcome_definition'}]
prohibited=[c for c in feature_columns if ('rain' in c or 'coverage' in c or 'observed' in c) and not (
    'lag' in c or 'prev' in c or c in ['expected_station_count','month_sin','month_cos'])]
if prohibited:raise AssertionError('Unverified same-day features: '+', '.join(prohibited))
unlagged_weather=[c for c in feature_columns if c.startswith(('rain_','coverage_','observed_'))
                  and not ('lag' in c or 'prev' in c)]
if unlagged_weather:
    raise AssertionError('Weather/coverage feature lacks an explicit lag or prior-window definition: '+', '.join(unlagged_weather))
assert not set(['reported_flood_events','flood_recorded','outcome_definition']).intersection(feature_columns)
assert (panel['rain_observed_days_prev3'].dropna().between(0,3)).all()
assert (panel['rain_observed_days_prev7'].dropna().between(0,7)).all()

mapping=frames['station']
station_present=mapping['in_rainfall_matrix'].astype(str).str.lower().eq('true')
id_verified=mapping['station_id_confirmed'].astype(str).str.lower().eq('true')
station_count=int(station_present.sum());id_count=int((station_present&id_verified).sum())
assert station_count==460 and id_count <= station_count
reported=frames['metrics'];test_metrics=reported[reported['period'].astype(str).str.contains('2025')]
assert len(test_metrics)>=8

integrity_rows=[
    ('Original data DOI',DOI,'base dataset metadata reference; any user-supplied extension must be documented separately'),
    ('Frozen audit version',VERSION,'two_stage_training_optional_and_audit'),
    ('Input CSV signature SHA-256',source_signature,'computed'),
    ('Historical daily region observations in panel',len(panel),'pass'),
    ('Post-2025 temporal rows',len(future_pred),'available' if len(future_pred) else 'not_provided'),
    ('Recorded positive region-days in panel',int(panel.flood_recorded.sum()),'pass'),
    ('2025 test region-days',len(pred),'pass'),
    ('2025 recorded positive region-days',int(pred.flood_recorded.sum()),'pass'),
    ('Test region count',pred.region_key.nunique(),'pass'),
    ('Rainfall-matrix station headers',station_count,'source_matrix'),
    ('Station headers with confirmed station ID',id_count,'not_all_confirmed'),
    ('Verified feature availability at issuance clock','unverified','date_only_flood_event'),
    ('2025 confirmatory holdout status','reused_in_v3_to_v8','exploratory_only'),
    ('External 2026 outcome data',len(future_pred),'prospective_candidate_present' if len(future_pred) else 'prospective_evaluation_pending'),
    ('Reported flood versus confirmed absence','no_report_is_not_confirmed_negative','label_bias_warning'),
    ('Model development history','2025 previously inspected','2025 metrics remain exploratory despite retraining; future outcomes must not influence model selection'),
    ('Frozen model bundle',str(globals().get('MODEL_BUNDLE_PATH','not_available')),'saved outside publication outputs when full training was run'),
]
# Tambahkan provenance yang dapat diaudit ulang, termasuk hash workbook bila training dijalankan pada sesi ini.
for source_name,digest in globals().get('SOURCE_SIGNATURE',{}).items():
    integrity_rows.append((f'Source workbook SHA-256: {source_name}',digest,'verified_during_source_load'))
integrity_rows.extend([
    ('Model pipeline version',PIPELINE_VERSION,'locked_version'),
    ('Model signature',globals().get('MODEL_SIGNATURE','not_available_in_audit_only_mode'),
     'exact-source-and-model-specification-signature'),
    ('Feature engineering version',FEATURE_ENGINEERING_VERSION,'locked_version'),
    ('Random seed',SEED,'locked'),
    ('FAST_TEST mode',FAST_TEST,'must_be_false_for_publication_grade_output'),
    ('Missingness replicates per scenario',ROBUSTNESS_REPS,'full_run_target_100'),
    ('Primary date-block bootstrap repetitions',N_BOOT_PRIMARY,'full_run_target_1000'),
    ('Calibration date-block bootstrap repetitions',N_CALIBRATION_BOOT,'full_run_target_600'),
    ('Historical calendar continuity',f'{len(expected_dates)} daily dates × {panel.region_key.nunique()} regions',
     'pass_complete_region_day_panel'),
    ('Outcome interpretation','recorded flood report','not independently verified physical flood truth')
])
integrity=pd.DataFrame(integrity_rows,columns=['audit_check','observed_value','decision'])
print('Panel and 2025 predictions agree:',len(check),'rows; 2025 recorded positives:',int(pred.flood_recorded.sum()))

### C7. Recompute all 2025 discrimination and calibration metrics

In [ ]:
# Menghitung ulang metrik dari probabilitas asli dan memeriksa angka yang dilaporkan.
MODELS={
 'Seasonal-region baseline':'p_seasonal_region_baseline',
 'Logistic Regression':'p_logistic_regression',
 'Random Forest':'p_random_forest',
 'HistGradientBoosting':'p_histgradientboosting',
 'Standard stacking':'p_standard_stacking',
 'Uncertainty-only stacking':'p_uncertainty_only_stacking',
 'Quality-only stacking':'p_quality_only_stacking',
 'Full uncertainty–quality stacking':'p_full_uncertainty_quality_stacking',
 'Stress-trained uncertainty–quality stacking':'p_stress_trained_uncertainty_quality_stacking'}
Y=pred.flood_recorded.to_numpy(dtype=int)
P={name:pred[col].to_numpy(dtype=float) for name,col in MODELS.items()}
assert all(np.isfinite(arr).all() and np.all((arr>=0)&(arr<=1)) for arr in P.values())

# Intersep dan kemiringan recalibration diukur, tidak dipakai untuk mengubah probabilitas.
def calibration_intercept_slope(y,p):
    z=np.log(np.clip(p,1e-8,1-1e-8)/np.clip(1-p,1e-8,1-1e-8))
    def objective(theta):
        score=theta[0]+theta[1]*z
        return np.logaddexp(0,score).sum()-np.dot(y,score)
    res=minimize(objective,x0=[0,1],method='L-BFGS-B',options={'maxiter':120})
    return (float(res.x[0]),float(res.x[1])) if res.success else (np.nan,np.nan)
base_brier=brier_score_loss(Y,P['Seasonal-region baseline'])
metrics=[]
for model,score in P.items():
    ap=average_precision_score(Y,score);auc=roc_auc_score(Y,score);bs=brier_score_loss(Y,score)
    reference=test_metrics[test_metrics.model.eq(model)]
    if len(reference)!=1:raise ValueError('Missing or duplicated 2025 metric for '+model)
    # Toleransi baseline lebih longgar karena nilai probabilitas seri yang terikat dapat berubah urutannya saat diekspor CSV.
    tolerance=2e-5 if model=='Seasonal-region baseline' else 1e-7
    assert abs(ap-float(reference.iloc[0]['average_precision']))<tolerance,(model,'AP mismatch')
    assert abs(bs-float(reference.iloc[0]['brier_score']))<1e-9,(model,'Brier mismatch')
    intercept,slope=calibration_intercept_slope(Y,score)
    metrics.append({'model':model,'test_n':len(Y),'test_positive_n':int(Y.sum()),'prevalence':Y.mean(),
                    'mean_predicted_probability':float(np.mean(score)),
                    'mean_probability_minus_observed_prevalence':float(np.mean(score)-np.mean(Y)),
                    'mean_probability_gap_percentage_points':float(100*(np.mean(score)-np.mean(Y))),
                    'average_precision':ap,'roc_auc':auc,'brier_score':bs,
                    'brier_skill_vs_seasonal_baseline':1-bs/base_brier,
                    'calibration_intercept_descriptive':intercept,
                    'calibration_slope_descriptive':slope,
                    'status':'exploratory_reused_2025_test'})
model_metrics=pd.DataFrame(metrics)
print(model_metrics[['model','average_precision','roc_auc','brier_score','brier_skill_vs_seasonal_baseline']].to_string(index=False))

### C9. Correct seven-day report-onset at-risk target

In [ ]:
# KOREKSI v11: eligibilitas onset harus ditentukan hanya dari TUJUH HARI SEBELUM hari target.
# Tidak boleh mengecualikan hanya kasus positif lanjutan sambil mempertahankan hari negatif pascalaporan.
ordered=panel.sort_values(['region_key','date']).copy()
ordered['previous_7d_positive_count']=ordered.groupby('region_key',sort=False)['flood_recorded'].transform(
    lambda s:s.shift(1).rolling(window=7,min_periods=7).sum())
onset_test=ordered.loc[ordered.date.dt.year==2025,[
    'date','region_key','flood_recorded','previous_7d_positive_count','coverage_lag1']].copy()
assert onset_test.previous_7d_positive_count.notna().all()
onset_test['at_risk_eligible_7d']=onset_test.previous_7d_positive_count.eq(0)
onset_test['is_report_onset_7d']=(onset_test.flood_recorded.eq(1) & onset_test.at_risk_eligible_7d).astype(int)
onset_test['is_report_continuation_7d']=(onset_test.flood_recorded.eq(1)&~onset_test.at_risk_eligible_7d).astype(int)
onset=pred.merge(onset_test[['date','region_key','at_risk_eligible_7d','is_report_onset_7d','is_report_continuation_7d']],
                 on=['date','region_key'],validate='one_to_one')
at_risk_mask=onset['at_risk_eligible_7d'].to_numpy(dtype=bool)
y_at_risk=onset.loc[at_risk_mask,'is_report_onset_7d'].to_numpy(dtype=int)
y_onset_all=onset['is_report_onset_7d'].to_numpy(dtype=int)
assert y_at_risk.sum()==179 and len(y_at_risk)==8286
assert int(onset.is_report_continuation_7d.sum())==117
assert int(onset.is_report_onset_7d.sum())+int(onset.is_report_continuation_7d.sum())==296
assert not (onset.at_risk_eligible_7d & onset.is_report_continuation_7d.astype(bool)).any()

# Pelaporan ganda dilakukan untuk membedakan dua estimand yang berbeda dan melacak bias seleksi v10.
legacy_mask=onset.is_report_continuation_7d.eq(0)
legacy_y=onset.loc[legacy_mask,'is_report_onset_7d'].to_numpy(dtype=int)
onset_metrics=[]
for name,col in MODELS.items():
    probabilities=onset[col].to_numpy(dtype=float)
    conditional_ap=average_precision_score(y_at_risk,probabilities[at_risk_mask])
    all_days_ap=average_precision_score(y_onset_all,probabilities)
    legacy_ap=average_precision_score(legacy_y,probabilities[legacy_mask])
    report_ap=float(model_metrics.set_index('model').loc[name,'average_precision'])
    onset_metrics.append({
        'model':name,'model_training_target':'recorded_flood_day',
        'original_report_day_ap':report_ap,
        'onset_at_risk_ap':conditional_ap,'onset_at_risk_roc_auc':roc_auc_score(y_at_risk,probabilities[at_risk_mask]),
        'onset_at_risk_n':len(y_at_risk),'onset_at_risk_positive_n':int(y_at_risk.sum()),
        'onset_at_risk_prevalence':float(y_at_risk.mean()),
        'onset_at_risk_ap_lift':float(conditional_ap/y_at_risk.mean()),
        'onset_all_region_days_ap':all_days_ap,'onset_all_region_days_n':len(y_onset_all),
        'onset_all_region_days_positive_n':int(y_onset_all.sum()),
        'onset_all_region_days_prevalence':float(y_onset_all.mean()),
        'legacy_v10_outcome_conditioned_ap_not_comparable':legacy_ap,
        'legacy_v10_n':int(legacy_mask.sum()),
        'prior_7d_ineligible_region_days':int((~at_risk_mask).sum()),
        'prior_7d_ineligible_negatives':int((~at_risk_mask & (onset.flood_recorded.to_numpy()==0)).sum()),
        'positive_continuation_days':int(onset.is_report_continuation_7d.sum()),
        'report_onset_is_hydrologically_verified':False,
        'evaluation_status':'exploratory_at_risk_conditioned_sensitivity_not_independent_holdout'})
onset_table=pd.DataFrame(onset_metrics)
print('2025 all days=',len(onset),'at-risk eligible days=',int(at_risk_mask.sum()),
      'at-risk positives=',int(y_at_risk.sum()),'excluded post-report days=',int((~at_risk_mask).sum()))
print(onset_table[['model','onset_at_risk_ap','onset_at_risk_ap_lift']].to_string(index=False))

### C11. Fixed daily warning capacity

In [ ]:
# Memeringkat wilayah di dalam setiap tanggal, bukan top-k dari keseluruhan tahun.
capacity_rows=[]
for name,col in MODELS.items():
    for k in [1,2,3]:
        total_hits=total_flags=0
        for day,sub in pred.groupby('date',sort=True):
            ranked=sub.sort_values([col,'region_key'],ascending=[False,True],kind='mergesort').head(k)
            total_hits+=int(ranked.flood_recorded.sum());total_flags+=len(ranked)
        capacity_rows.append({'model':name,'regions_flagged_per_day':k,'days':pred.date.nunique(),
            'total_flags':total_flags,'recorded_positive_region_days_detected':total_hits,
            'false_alerts':total_flags-total_hits,
            'precision':total_hits/total_flags,'recall':total_hits/int(Y.sum()),
            'lift_vs_prevalence':(total_hits/total_flags)/Y.mean(),
            'fraction_of_regions_flagged_per_day':k/pred.region_key.nunique(),
            'status':'retrospective_capacity_sensitivity_not_operational_validation'})
capacity_table=pd.DataFrame(capacity_rows)

### C13. Paired date-block bootstrap and onset sensitivity

In [ ]:
# Blok tanggal menjaga korelasi wilayah dalam hari yang sama dan sebagian dependensi antarhari.
days=pd.Index(sorted(pred.date.unique()));n_days=len(days)
assert n_days==365
idx_by_day=[np.flatnonzero(pred.date.to_numpy()==day) for day in days]
COMPARS=[('Stress vs standard','Stress-trained uncertainty–quality stacking','Standard stacking'),
         ('Stress vs HistGB','Stress-trained uncertainty–quality stacking','HistGradientBoosting'),
         ('Stress vs full','Stress-trained uncertainty–quality stacking','Full uncertainty–quality stacking'),
         ('Full vs standard','Full uncertainty–quality stacking','Standard stacking')]
boot_rows=[]
for block_days in BLOCK_LENGTHS:
    reps=N_BOOT_PRIMARY if block_days==7 else N_BOOT_SENSITIVITY
    rng=np.random.default_rng(RANDOM_SEED+block_days)
    draw_records={title:([],[]) for title,_,_ in COMPARS}
    for repeat in range(reps):
        day_ids=sample_non_circular_blocks(n_days, block_days, rng)
        sample=np.concatenate([idx_by_day[int(i)] for i in day_ids])
        yy=Y[sample]
        if yy.min()==yy.max():continue
        for title,a,b in COMPARS:
            pa=P[a][sample];pb=P[b][sample]
            delta_ap=average_precision_score(yy,pa)-average_precision_score(yy,pb)
            improvement_bs=np.mean((pb-yy)**2-(pa-yy)**2)
            draw_records[title][0].append(delta_ap);draw_records[title][1].append(improvement_bs)
    for title,a,b in COMPARS:
        ap_arr=np.asarray(draw_records[title][0]);bs_arr=np.asarray(draw_records[title][1])
        if len(ap_arr)<int(.9*reps):raise RuntimeError('Insufficient valid bootstrap samples; inspect class prevalence')
        alpha=.05;adjusted_alpha=alpha/(len(COMPARS)*len(BLOCK_LENGTHS))
        row={'comparison':title,'model_a':a,'model_b':b,'block_days':block_days,'bootstrap_reps':len(ap_arr),
             'observed_delta_ap':average_precision_score(Y,P[a])-average_precision_score(Y,P[b]),
             'ap_ci95_low':np.quantile(ap_arr,.025),'ap_ci95_high':np.quantile(ap_arr,.975),
             'ap_bonferroni_sensitivity_low':np.quantile(ap_arr,adjusted_alpha/2),
             'ap_bonferroni_sensitivity_high':np.quantile(ap_arr,1-adjusted_alpha/2),
             'observed_brier_improvement':np.mean((P[b]-Y)**2-(P[a]-Y)**2),
             'brier_ci95_low':np.quantile(bs_arr,.025),'brier_ci95_high':np.quantile(bs_arr,.975),
             'brier_bonferroni_sensitivity_low':np.quantile(bs_arr,adjusted_alpha/2),
             'brier_bonferroni_sensitivity_high':np.quantile(bs_arr,1-adjusted_alpha/2),
             'evaluation_status':'exploratory_reused_2025_test_non_circular_blocks',
             'bootstrap_method':'non-circular moving-block bootstrap',
             'multiplicity_adjustment_scope':'Bonferroni sensitivity across 4 contrasts x 3 block lengths'}
        boot_rows.append(row)
bootstrap_table=pd.DataFrame(boot_rows)
# Inferensi eksploratif tambahan: target baru 'onset at-risk' mempunyai denominasi tersendiri.
# Resampling selalu berdasarkan tanggal kalender; semua wilayah yang eligible ikut bersama.
# Tidak menghitung Brier onset karena probabilitas model dilatih untuk hari laporan biasa.
for old_row in boot_rows:
    old_row['outcome_target']='recorded_flood_day'

onset_dates=np.asarray(sorted(pd.unique(onset.date)))
assert len(onset_dates)==365
at_risk_rows=onset.loc[at_risk_mask].reset_index(drop=True)
onset_date_indices=[np.flatnonzero(at_risk_rows.date.to_numpy()==day) for day in onset_dates]
onset_y=at_risk_rows.is_report_onset_7d.to_numpy(dtype=int)
assert len(onset_y)==8286 and int(onset_y.sum())==179
onset_comparators=[('At-risk onset: Stress vs standard','Standard stacking'),
                   ('At-risk onset: Stress vs HistGB','HistGradientBoosting')]
for block_days in BLOCK_LENGTHS:
    reps=N_BOOT_PRIMARY if block_days==7 else N_BOOT_SENSITIVITY
    rng=np.random.default_rng(RANDOM_SEED+901+block_days)
    total_days=len(onset_dates)
    pi=at_risk_rows[MODELS['Stress-trained uncertainty–quality stacking']].to_numpy(dtype=float)
    for label,benchmark in onset_comparators:
        pb=at_risk_rows[MODELS[benchmark]].to_numpy(dtype=float)
        delta_point=average_precision_score(onset_y,pi)-average_precision_score(onset_y,pb)
        effects=[]
        for _ in range(reps):
            days_resampled=sample_non_circular_blocks(total_days, block_days, rng)
            ids=np.concatenate([onset_date_indices[int(j)] for j in days_resampled])
            yy=onset_y[ids]
            if len(yy)==0 or yy.min()==yy.max():continue
            effects.append(average_precision_score(yy,pi[ids])-average_precision_score(yy,pb[ids]))
        if len(effects)<int(.9*reps):raise RuntimeError('Insufficient onset bootstrap replicates')
        boot_rows.append({'comparison':label,
            'model_a':'Stress-trained uncertainty–quality stacking','model_b':benchmark,
            'block_days':block_days,'bootstrap_reps':len(effects),
            'observed_delta_ap':delta_point,'ap_ci95_low':float(np.quantile(effects,.025)),
            'ap_ci95_high':float(np.quantile(effects,.975)),
            'ap_bonferroni_sensitivity_low':np.nan,'ap_bonferroni_sensitivity_high':np.nan,
            'observed_brier_improvement':np.nan,'brier_ci95_low':np.nan,'brier_ci95_high':np.nan,
            'brier_bonferroni_sensitivity_low':np.nan,'brier_bonferroni_sensitivity_high':np.nan,
            'outcome_target':'conditional_7day_report_onset_at_risk',
            'evaluation_status':'exploratory_posthoc_onset_definition_no_brier',
            'bootstrap_method':'non-circular moving-block bootstrap',
            'multiplicity_adjustment_scope':'Not adjusted; post-hoc onset target sensitivity'})
bootstrap_table=pd.DataFrame(boot_rows)
print('Completed bootstrap comparisons:',len(bootstrap_table))

### C15. Paired station-level missingness robustness

In [ ]:
# Data eksperimen asli tidak diubah; komparasi dijodohkan pada gangguan yang identik.
r=frames['robustness'].copy()
assert {'mechanism','target_additional_missing_fraction','replicate','model','average_precision','brier_score'}.issubset(r.columns)
r=r[r.mechanism.ne('Original')].copy()
index=['mechanism','target_additional_missing_fraction','replicate']
assert r.duplicated(index+['model']).sum()==0
p_ap=r.pivot(index=index,columns='model',values='average_precision')
p_bs=r.pivot(index=index,columns='model',values='brier_score')
for model in ['Stress-trained uncertainty–quality stacking','Standard stacking','HistGradientBoosting']:
    assert p_ap[model].notna().all()
robust_rows=[]
for (mechanism,severity),group in p_ap.groupby(level=[0,1]):
    ix=group.index
    for comparator in ['Standard stacking','HistGradientBoosting']:
        delta=(p_ap.loc[ix,'Stress-trained uncertainty–quality stacking']-p_ap.loc[ix,comparator]).to_numpy()
        improve=(p_bs.loc[ix,comparator]-p_bs.loc[ix,'Stress-trained uncertainty–quality stacking']).to_numpy()
        robust_rows.append({'mechanism':mechanism,'additional_missing_target_fraction':severity,
            'comparator':comparator,'replicates':len(delta),'delta_ap_mean':delta.mean(),
            'delta_ap_q025_simulation':np.quantile(delta,.025),'delta_ap_q975_simulation':np.quantile(delta,.975),
            'ap_better_fraction':float((delta>0).mean()),'brier_improvement_mean':improve.mean(),
            'brier_q025_simulation':np.quantile(improve,.025),'brier_q975_simulation':np.quantile(improve,.975),
            'brier_better_fraction':float((improve>0).mean()),
            'uncertainty_qualification':'simulation spread, not statistical confidence interval'})
robustness_summary=pd.DataFrame(robust_rows)
print('Paired disturbance groups:',len(robustness_summary))

### C17. Regional statistical support and cluster-aware calibration

In [ ]:
# Menggabungkan ringkasan wilayah dan audit dukungan positif, bukan meranking wilayah yang datanya sangat sedikit.
region_rows=[]
for region,g in pred.groupby('region_key',sort=True):
    yy=g.flood_recorded.to_numpy(dtype=int)
    cov=pd.to_numeric(g.coverage_lag1,errors='coerce')
    row={'region_key':region,'test_region_days':len(g),'test_positive_n':int(yy.sum()),
         'mean_lag1_station_coverage':float(cov.mean()),
         'zero_coverage_days':int(cov.eq(0).sum()),'below_50pct_coverage_days':int(cov.lt(.5).sum()),
         'ap_reliability_flag':'descriptive_only' if int(yy.sum())>=10 else 'insufficient_positive_support'}
    for name,key in [('stress','p_stress_trained_uncertainty_quality_stacking'),
                     ('standard','p_standard_stacking'),('histgb','p_histgradientboosting')]:
        pr=g[key].to_numpy(dtype=float)
        row[f'brier_{name}']=brier_score_loss(yy,pr)
        row[f'ap_{name}']=average_precision_score(yy,pr) if yy.sum()>=10 else np.nan
    region_rows.append(row)
regions_table=pd.DataFrame(region_rows)

# Equal-frequency deciles by rank, bin IDs deterministic, Wilson interval approximate iid.
def wilson_interval(k,n,z=1.95996398454):
    phat=k/n;den=1+z*z/n;centre=(phat+z*z/(2*n))/den
    half=z*np.sqrt(phat*(1-phat)/n+z*z/(4*n*n))/den
    return centre-half,centre+half
calibration_rows=[]
for name in ['HistGradientBoosting','Standard stacking','Full uncertainty–quality stacking',
             'Stress-trained uncertainty–quality stacking']:
    order=np.argsort(P[name],kind='mergesort')
    for bin_no,ids in enumerate(np.array_split(order,10),1):
        positives=int(Y[ids].sum());lo,hi=wilson_interval(positives,len(ids))
        calibration_rows.append({'model':name,'bin':bin_no,'n':len(ids),'positives':positives,
                                 'mean_probability':float(P[name][ids].mean()),
                                 'observed_report_fraction':float(Y[ids].mean()),
                                 'wilson95_low_naive_iid':lo,'wilson95_high_naive_iid':hi,
                                 'interval_limitation':'iid Wilson; ignores temporal and spatial dependence'})
calibration_table=pd.DataFrame(calibration_rows)

# Interval bootstrap blok tanggal, mempertahankan seluruh wilayah pada tanggal yang sama.
# Batas bin ditetapkan dari urutan prediksi ASLI, bukan diperbarui setiap replikasi.
# Ini mengukur variasi temporal-kelompok (bukan kepastian kalibrasi konfirmatori).
rng_cal=np.random.default_rng(RANDOM_SEED+111)
cal_days=np.array(sorted(pd.unique(pred.date)))
assert len(cal_days)==365
cal_day_idx=pd.Index(cal_days).get_indexer(pred.date.to_numpy())
cal_ci={}
for name in ['HistGradientBoosting','Standard stacking','Full uncertainty–quality stacking',
             'Stress-trained uncertainty–quality stacking']:
    bin_of_row=np.empty(len(Y),dtype=int)
    for b,ids in enumerate(np.array_split(np.argsort(P[name],kind='mergesort'),10)):
        bin_of_row[ids]=b
    day_bin_counts=np.zeros((365,10),dtype=int)
    day_bin_pos=np.zeros((365,10),dtype=int)
    np.add.at(day_bin_counts,(cal_day_idx,bin_of_row),1)
    np.add.at(day_bin_pos,(cal_day_idx,bin_of_row),Y)
    boot_est=np.full((N_CALIBRATION_BOOT,10),np.nan,dtype=float)
    for rep in range(N_CALIBRATION_BOOT):
        selected=sample_non_circular_blocks(365, 7, rng_cal)
        bn=day_bin_counts[selected].sum(axis=0)
        bk=day_bin_pos[selected].sum(axis=0)
        boot_est[rep]=np.divide(bk,bn,out=np.full(10,np.nan),where=bn>0)
    for b in range(10):
        vals=boot_est[:,b];vals=vals[np.isfinite(vals)]
        if len(vals)<int(.9*N_CALIBRATION_BOOT):raise RuntimeError('Insufficient bin bootstrap samples')
        cal_ci[(name,b+1)]=(float(np.quantile(vals,.025)),float(np.quantile(vals,.975)),len(vals))
for r in calibration_rows:
    lo,hi,reps=cal_ci[(r['model'],r['bin'])]
    r['date_block7_boot95_low']=lo
    r['date_block7_boot95_high']=hi
    r['date_block7_boot_reps']=reps
    r['bootstrap_limitation']='Non-circular 7-day moving-block bootstrap; fixed calibration bins; repeated 2025 test use'
calibration_table=pd.DataFrame(calibration_rows)

# Tambahkan tabel ringkasan kualitas per wilayah-tahun dengan faktor pendukung kesimpulan.
quality['mean_available_station_fraction_lag1']=pd.to_numeric(quality['mean_available_station_fraction_lag1'],errors='coerce')
quality_audit=quality[['region_key','year','region_days','positive_region_days',
                       'mean_available_station_fraction_lag1','lag1_no_station_days',
                       'lag1_coverage_below_half_days']].copy()
quality_audit['recorded_positive_rate']=quality_audit.positive_region_days / quality_audit.region_days
quality_audit['evaluation_period_role']=np.select(
    [quality_audit.year.le(2023),quality_audit.year.eq(2024),quality_audit.year.eq(2025)],
    ['training','validation_and_calibration','exploratory_test_reused_during_development'],
    default='outside_source_period')
quality_audit['data_quality_flag']=np.where(quality_audit.mean_available_station_fraction_lag1.lt(.5),
                                          'low_coverage_below_50pct','check')

### C19. Editorial readiness gates

In [ ]:
# Kriteria transparan dipakai untuk menilai keterbatasan naskah, bukan untuk mengoptimasi skor.
AP_stress=model_metrics.set_index('model').loc['Stress-trained uncertainty–quality stacking','average_precision']
BS_stress=model_metrics.set_index('model').loc['Stress-trained uncertainty–quality stacking','brier_score']
ap_robust_frac=robustness_summary[robustness_summary.comparator.eq('Standard stacking')]['ap_better_fraction'].mean()
missingness_reps_min=int(robustness_summary['replicates'].min()) if len(robustness_summary) else 0
regional_supported=int((regions_table.test_positive_n>=10).sum())

# Label setiap baris bootstrap agar hasil 2025 tidak tercampur dengan onset atau evaluasi masa depan.
if 'evaluation_period' not in bootstrap_table.columns:
    bootstrap_table['evaluation_period']=np.where(
        bootstrap_table.get('outcome_target',pd.Series('recorded_flood_day',index=bootstrap_table.index)).eq('recorded_flood_day'),
        'exploratory_2025','exploratory_2025_onset_sensitivity')

# Evaluasi temporal pasca-2025 bersifat opsional. Jika workbook sumber diperluas, prediksi tetap berasal
# dari model yang dilatih hanya pada 2020–2023 dan kalibrator yang dipasang hanya pada 2024.
prospective_predictions=future_pred.copy() if isinstance(future_pred,pd.DataFrame) else pd.DataFrame()
prospective_metrics=pd.DataFrame()
future_boot_rows=[]
future_gate_status='MISSING'
future_gate_reason='Tidak tersedia observasi setelah 2025; hasil 2025 tetap eksploratif.'
future_signature_ok=False
if not prospective_predictions.empty:
    prospective_predictions['date']=pd.to_datetime(prospective_predictions['date'],errors='raise')
    required_future={'date','region_key','flood_recorded','model_signature'}
    missing_future=required_future-set(prospective_predictions.columns)
    if missing_future:
        raise ValueError('Prediksi temporal masa depan kehilangan kolom wajib: '+str(sorted(missing_future)))
    if prospective_predictions['date'].dt.year.min()<2026:
        raise ValueError('Prospective holdout harus hanya mencakup tanggal mulai 2026.')
    if prospective_predictions.duplicated(['date','region_key']).any():
        raise ValueError('Duplikasi tanggal-wilayah pada prediksi masa depan.')
    if not set(prospective_predictions['flood_recorded'].dropna().unique()).issubset({0,1}):
        raise ValueError('Label masa depan harus berupa 0/1.')
    expected_model_signature=globals().get('MODEL_SIGNATURE',None)
    future_signature_ok=(expected_model_signature is not None and
        prospective_predictions['model_signature'].astype(str).nunique()==1 and
        prospective_predictions['model_signature'].astype(str).iloc[0]==str(expected_model_signature))
    if not future_signature_ok:
        future_gate_status='REJECTED_SIGNATURE_MISMATCH'
        future_gate_reason='Signature model pada prediksi masa depan tidak cocok dengan model terkunci.'
    else:
        y_future_eval=prospective_predictions['flood_recorded'].to_numpy(dtype=int)
        n_future_dates=prospective_predictions['date'].nunique()
        future_prevalence=float(y_future_eval.mean()) if len(y_future_eval) else np.nan
        future_rows=[]
        for model_name,prob_col in MODELS.items():
            if prob_col not in prospective_predictions.columns:
                raise ValueError('Kolom probabilitas masa depan tidak ditemukan: '+prob_col)
            p_future_eval=prospective_predictions[prob_col].to_numpy(dtype=float)
            if not np.isfinite(p_future_eval).all() or np.any((p_future_eval<0)|(p_future_eval>1)):
                raise ValueError('Probabilitas masa depan harus finite dan berada pada [0,1].')
            varied=np.unique(y_future_eval).size==2
            ap_future=float(average_precision_score(y_future_eval,p_future_eval)) if varied else np.nan
            auc_future=float(roc_auc_score(y_future_eval,p_future_eval)) if varied else np.nan
            brier_future=float(brier_score_loss(y_future_eval,p_future_eval))
            logloss_future=float(log_loss(y_future_eval,np.clip(p_future_eval,1e-6,1-1e-6),labels=[0,1]))
            ece_future=0.0
            bins=np.linspace(0,1,11)
            bin_id=np.minimum(np.digitize(p_future_eval,bins[1:-1]),9)
            for bin_no in range(10):
                use=bin_id==bin_no
                if use.any():
                    ece_future+=(use.mean()*abs(y_future_eval[use].mean()-p_future_eval[use].mean()))
            future_rows.append({
                'model':model_name,'period':'prospective_2026plus',
                'test_n':len(y_future_eval),'test_positive_n':int(y_future_eval.sum()),
                'prevalence':future_prevalence,
                'mean_predicted_probability':float(p_future_eval.mean()),
                'mean_probability_minus_observed_prevalence':float(p_future_eval.mean()-future_prevalence),
                'mean_probability_gap_percentage_points':float(100*(p_future_eval.mean()-future_prevalence)),
                'average_precision':ap_future,'roc_auc':auc_future,'brier_score':brier_future,
                'log_loss':logloss_future,'ece_10bins':float(ece_future),
                'brier_skill_vs_seasonal_baseline':np.nan,
                'calibration_intercept_descriptive':np.nan,'calibration_slope_descriptive':np.nan,
                'status':'prospective_candidate_requires_independence_attestation'})
        prospective_metrics=pd.DataFrame(future_rows)
        # Brier skill relatif au seasonal-region baseline dari periode yang sama.
        base_future=prospective_metrics.loc[
            prospective_metrics.model.eq('Seasonal-region baseline'),'brier_score'].iloc[0]
        prospective_metrics['brier_skill_vs_seasonal_baseline']=1-prospective_metrics.brier_score/base_future if base_future>0 else np.nan

        # Bootstrap berpasangan per tanggal hanya dijalankan jika ada variasi label dan rentang waktu memadai.
        future_days=pd.Index(sorted(prospective_predictions['date'].unique()))
        n_future_days=len(future_days)
        if n_future_days>=30 and np.unique(y_future_eval).size==2:
            idx_future_by_day={pd.Timestamp(day):np.flatnonzero(
                prospective_predictions['date'].to_numpy()==day) for day in future_days}
            available_blocks=[b for b in BLOCK_LENGTHS if b<=n_future_days]
            for block_days in available_blocks:
                reps=N_BOOT_PRIMARY if block_days==7 else N_BOOT_SENSITIVITY
                rng=np.random.default_rng(RANDOM_SEED+5000+block_days)
                samples={title:([],[]) for title,_,_ in COMPARS}
                for _ in range(reps):
                    selected_days=sample_non_circular_blocks(n_future_days,block_days,rng)
                    ids=np.concatenate([idx_future_by_day[pd.Timestamp(future_days[int(j)])] for j in selected_days])
                    yy=y_future_eval[ids]
                    if np.unique(yy).size<2: continue
                    for title,model_a,model_b in COMPARS:
                        pa=prospective_predictions[MODELS[model_a]].to_numpy(dtype=float)[ids]
                        pb=prospective_predictions[MODELS[model_b]].to_numpy(dtype=float)[ids]
                        samples[title][0].append(average_precision_score(yy,pa)-average_precision_score(yy,pb))
                        samples[title][1].append(np.mean((pb-yy)**2-(pa-yy)**2))
                alpha=.05/(len(COMPARS)*max(1,len(available_blocks)))
                for title,model_a,model_b in COMPARS:
                    arr_ap=np.asarray(samples[title][0]);arr_bs=np.asarray(samples[title][1])
                    if len(arr_ap)<int(.9*reps): continue
                    future_boot_rows.append({
                        'comparison':title,'model_a':model_a,'model_b':model_b,'block_days':block_days,
                        'bootstrap_reps':len(arr_ap),
                        'observed_delta_ap':average_precision_score(y_future_eval,
                            prospective_predictions[MODELS[model_a]].to_numpy(dtype=float))-
                            average_precision_score(y_future_eval,prospective_predictions[MODELS[model_b]].to_numpy(dtype=float)),
                        'ap_ci95_low':float(np.quantile(arr_ap,.025)),'ap_ci95_high':float(np.quantile(arr_ap,.975)),
                        'ap_bonferroni_sensitivity_low':float(np.quantile(arr_ap,alpha/2)),
                        'ap_bonferroni_sensitivity_high':float(np.quantile(arr_ap,1-alpha/2)),
                        'observed_brier_improvement':float(np.mean(
                            (prospective_predictions[MODELS[model_b]].to_numpy(dtype=float)-y_future_eval)**2-
                            (prospective_predictions[MODELS[model_a]].to_numpy(dtype=float)-y_future_eval)**2)),
                        'brier_ci95_low':float(np.quantile(arr_bs,.025)),'brier_ci95_high':float(np.quantile(arr_bs,.975)),
                        'brier_bonferroni_sensitivity_low':float(np.quantile(arr_bs,alpha/2)),
                        'brier_bonferroni_sensitivity_high':float(np.quantile(arr_bs,1-alpha/2)),
                        'outcome_target':'recorded_flood_day',
                        'evaluation_status':'prospective_temporal_holdout_candidate',
                        'bootstrap_method':'non-circular moving-block bootstrap',
                        'multiplicity_adjustment_scope':'Bonferroni across model contrasts and available block lengths',
                        'evaluation_period':'prospective_2026plus'})
            if future_boot_rows:
                bootstrap_table=pd.concat([bootstrap_table,pd.DataFrame(future_boot_rows)],ignore_index=True,sort=False)

        # Untuk klaim confirmatory dibutuhkan satu siklus tahunan, variasi label, signature cocok,
        # dan attestasi eksplisit bahwa hasil ini tidak dipakai untuk memilih model.
        if n_future_days<365:
            future_gate_status='PARTIAL_PERIOD_NOT_CONFIRMATORY'
            future_gate_reason=f'Hanya {n_future_days} hari unik pasca-2025; belum mencakup satu siklus tahunan penuh.'
        elif np.unique(y_future_eval).size<2:
            future_gate_status='INSUFFICIENT_OUTCOME_VARIATION'
            future_gate_reason='Label masa depan tidak memiliki kelas positif dan negatif.'
        elif not ATTEST_FUTURE_DATA_UNSEEN_DURING_MODEL_DEVELOPMENT:
            future_gate_status='ATTESTATION_REQUIRED'
            future_gate_reason='Data pasca-2025 tersedia, tetapi independensi belum diattestasi oleh peneliti.'
        else:
            future_gate_status='AVAILABLE_WITH_SCOPE_LIMITATIONS'
            future_gate_reason='Terdapat satu siklus tahunan dan attestasi independensi; target tetap laporan banjir, bukan kebenaran fisik kejadian.'

bs_effect=bootstrap_table.loc[(bootstrap_table.comparison=='Stress vs standard')&
                             (bootstrap_table.block_days==7)&
                             (bootstrap_table.evaluation_period=='exploratory_2025')].iloc[0]
annual = quality_audit.groupby('year', as_index=False).agg(
    region_days=('region_days','sum'),
    positives=('positive_region_days','sum'),
    mean_coverage=('mean_available_station_fraction_lag1','mean'))
annual['prevalence'] = annual.positives / annual.region_days
train_rate = float(annual.loc[annual.year.le(2023), 'positives'].sum() /
                   annual.loc[annual.year.le(2023), 'region_days'].sum())
val_rate = float(annual.loc[annual.year.eq(2024), 'prevalence'].iloc[0])
test_rate = float(annual.loc[annual.year.eq(2025), 'prevalence'].iloc[0])
train_cov = float(annual.loc[annual.year.le(2023)].mean_coverage.mean())
val_cov = float(annual.loc[annual.year.eq(2024), 'mean_coverage'].iloc[0])
test_cov = float(annual.loc[annual.year.eq(2025), 'mean_coverage'].iloc[0])
std_row = model_metrics.set_index('model').loc['Standard stacking']
brier_relative_improvement = (float(std_row.brier_score) - float(BS_stress)) / float(std_row.brier_score)
editorial_rows=[
 ('Exact dataset version and source DOI','PASS',DOI),
 ('Recomputed 2025 model AP / Brier','PASS','Metrics recomputed from row-level probabilities and match saved values'),
 ('Future observation leakage','REQUIRES_METHODS_ATTESTATION','All rainfall features use lags, but issue-time availability and report timestamp are unverified'),
 ('Negative label validity','LIMITATION','No report does not imply physically confirmed absence of a flood'),
 ('Prospective at-risk onset denominator corrected','PASS',f'Eligible {len(y_at_risk):,} region-days; {int(y_at_risk.sum())} recorded onset positives; {int((~at_risk_mask).sum())} ineligible post-report region-days'),
 ('At-risk onset paired AP inference','EXPLORATORY_ONLY','Non-circular 7/14/30-day block sensitivity; confidence intervals include zero for key comparisons'),
 ('Evidence against strongest single learner','NONCONCLUSIVE',
  'Stress AP gain over HistGradientBoosting is 0.00455; nominal 7-day interval barely excludes zero, but 14/30-day and multiplicity-adjusted intervals include zero'),
 ('Robustness of AP under station missingness','CONTRADICTED_AS_UNIVERSAL_CLAIM',
  f'Across paired missingness replicates, the mean fraction with stress AP > standard was {ap_robust_frac:.1%}'),
 ('Missingness simulation replication count','PASS' if missingness_reps_min>=100 else 'BELOW_TARGET',
  f'Minimum replicates per mechanism-severity-model group: {missingness_reps_min}; target is 100 for the full frozen-evidence run'),
 ('Cluster-dependent calibration-bin uncertainty','EXPLORATORY_ONLY','Non-circular 7-day date-block bootstrap; fixed bins and reused test remain limitations'),
 ('Brier / probabilistic performance','EXPLORATORY_POSITIVE',
  f'Stress Brier={BS_stress:.6f}; relative improvement vs standard={brier_relative_improvement:.2%}; 2025 remains exploratory'),
 ('Mean-risk calibration under prevalence shift','UNDERPREDICTION_RISK',
  f'Stress mean predicted probability={pred[MODELS["Stress-trained uncertainty–quality stacking"]].mean():.2%} vs observed report prevalence={Y.mean():.2%}; gap={(pred[MODELS["Stress-trained uncertainty–quality stacking"]].mean()-Y.mean())*100:.3f} percentage points'),
 ('Temporal reporting and coverage drift','MATERIAL_RISK',
  f'Reported-positive rate train={train_rate:.2%}, 2024={val_rate:.2%}, 2025={test_rate:.2%}; mean station coverage train={train_cov:.2%}, 2024={val_cov:.2%}, 2025={test_cov:.2%}'),
 ('Regional subgroup support','LIMITED',f'Only {regional_supported}/27 regions have >=10 test positives'),
 ('Independent post-2025 temporal holdout',future_gate_status,future_gate_reason),
 ('Prospective validation after model lock',future_gate_status,future_gate_reason),
 ('Future model-signature consistency','PASS' if future_signature_ok else ('NOT_APPLICABLE' if prospective_predictions.empty else 'FAIL'),
  'Future predictions must carry the same frozen model signature as the fitted model bundle'),
 ('Future data independence attestation','PASS' if ATTEST_FUTURE_DATA_UNSEEN_DURING_MODEL_DEVELOPMENT else 'NOT_ATTESTED',
  'Set the attestation flag to True only when post-2025 outcomes were not used to design, select, or tune the model'),
 ('Reported label onset time and lead horizon','MISSING','Daily labels lack verified intra-day onset time; do not claim a 24-hour warning horizon'),
 ('External rainfall station map and regional matching','AUDIT_REQUIRED','458 of 460 station headers map to confirmed station IDs; document remaining stations and metadata limitations'),
 ('Novelty against current literature','NOT_ASSESSED','A systematic comparison against recent uncertainty-aware ensembles and flood-report forecasting studies is still required'),
 ('Data and code availability for submission','INCOMPLETE','Dataset DOI is known; archive the final notebook, environment, model artifact, data dictionary, and versioned outputs in a persistent repository'),
 ('Operational decision utility','NOT_VALIDATED','Top-k alerts are retrospective; evaluate false-alert cost, missed-event cost, and actual lead time before operational claims'),
 ('Scientific claim permissible','QUALIFIED','Current evidence supports an exploratory calibration/Brier signal, not a confirmed general discrimination advantage'),
 ('Submission decision','NOT_READY','Resolve outcome validity, temporal drift, independent validation, station provenance, and novelty before submission')]
editorial_table=pd.DataFrame(editorial_rows,columns=['evidence_gate','decision','reason_or_result'])

In [ ]:
# v16 publication-claim guardrails.
# Audit ini tidak mengubah skor atau memilih model berdasarkan test set.
# Ia menambahkan keputusan eksplisit ke tabel editorial sebelum ekspor 10 CSV.

def _v16_add_gate(name, decision, reason):
    global editorial_table
    if name not in set(editorial_table["evidence_gate"].astype(str)):
        editorial_table = pd.concat(
            [editorial_table, pd.DataFrame([{
                "evidence_gate": name,
                "decision": decision,
                "reason_or_result": reason
            }])],
            ignore_index=True
        )

# 1) Klaim ketahanan AP terhadap standard stacking pada missingness.
_v16_std = robustness_summary.loc[
    robustness_summary["comparator"].eq("Standard stacking")
]
_v16_ap_better_fraction = (
    float(_v16_std["ap_better_fraction"].mean()) if len(_v16_std) else float("nan")
)
_v16_robust_decision = (
    "SUPPORTED" if len(_v16_std) and _v16_ap_better_fraction >= 0.5
    else "NOT_SUPPORTED"
)
_v16_add_gate(
    "Claim audit: universal AP robustness under missingness",
    _v16_robust_decision,
    (
        f"Across paired mechanism-severity groups, stress-trained AP exceeded "
        f"standard stacking in {_v16_ap_better_fraction:.1%} of replicates on average. "
        "Do not claim universal AP robustness when this proportion is below 50%."
        if len(_v16_std) else
        "No paired standard-stacking missingness comparison was available."
    )
)

# 2) Klaim superioritas AP diperiksa pada semua panjang blok dan koreksi multipel.
_v16_ap_boot = bootstrap_table.loc[
    bootstrap_table["comparison"].isin(["Stress vs standard", "Stress vs HistGB"])
].copy()
_v16_raw_ci_all_positive = bool(
    len(_v16_ap_boot) and (_v16_ap_boot["ap_ci95_low"] > 0).all()
)
_v16_adjusted_ci_all_positive = bool(
    len(_v16_ap_boot) and (_v16_ap_boot["ap_bonferroni_sensitivity_low"] > 0).all()
)
_v16_ap_decision = (
    "SUPPORTED" if _v16_raw_ci_all_positive and _v16_adjusted_ci_all_positive
    else "NOT_CONCLUSIVE"
)
_v16_add_gate(
    "Claim audit: confirmatory AP superiority across block lengths",
    _v16_ap_decision,
    (
        "Confirmatory superiority requires lower confidence bounds above zero across "
        "all tested block lengths and the multiplicity sensitivity interval. "
        f"Raw lower bounds all positive={_v16_raw_ci_all_positive}; "
        f"adjusted lower bounds all positive={_v16_adjusted_ci_all_positive}. "
        "Current 2025 results remain exploratory."
    )
)

# 3) Validasi temporal independen harus berasal dari label pasca-2025 dan model beku.
_v16_future_valid = bool(
    isinstance(prospective_metrics, pd.DataFrame)
    and not prospective_metrics.empty
    and str(future_gate_status).upper() == "PASS"
    and bool(future_signature_ok)
)
_v16_add_gate(
    "Claim audit: independent post-2025 temporal validation",
    "PASS" if _v16_future_valid else "MISSING",
    (
        "Post-2025 predictions use a frozen model signature and pass the prospective gate."
        if _v16_future_valid else
        "No qualifying independent post-2025 temporal holdout is available. "
        "Do not describe 2025 as an independent confirmatory test."
    )
)

# 4) Label negatif harus ditafsirkan sebagai tidak ada laporan, bukan banjir terverifikasi tidak terjadi.
_v16_add_gate(
    "Claim audit: interpretation of negative labels",
    "LIMITATION",
    "A region-day without a flood report is not confirmed flood absence. "
    "Use recorded-report terminology unless an independently verified event inventory is obtained."
)

# 5) Final editorial gate is conservative and cannot be overridden by a high model score.
_v16_add_gate(
    "Claim audit: overall Teknika SINTA-2 submission readiness",
    "NOT_READY" if not _v16_future_valid else "REVIEW_REQUIRED",
    "The 2025 test was repeatedly inspected, outcome labels are report-based, station mapping is incomplete, "
    "and novelty against current literature remains to be assessed. A higher AP alone cannot close these gaps."
)

print("v16 claim guardrails appended. These gates document evidence limits; they do not alter model scores.")


In [ ]:
# v17 Teknika SINTA-2 submission gates.
# These checks distinguish computational evidence from requirements that need a manuscript or external check.
# They do not tune models or alter test-set scores.
def _v17_add_teknika_gate(name, decision, reason):
    global editorial_table
    if name not in set(editorial_table["evidence_gate"].astype(str)):
        editorial_table = pd.concat(
            [editorial_table, pd.DataFrame([{
                "evidence_gate": name,
                "decision": decision,
                "reason_or_result": reason
            }])],
            ignore_index=True
        )

_v17_add_teknika_gate(
    "Teknika fit: ICT and machine-learning scope",
    "ALIGNED",
    "Flood-report prediction, machine learning, missing-data robustness, and probabilistic evaluation fit the journal's ICT scope."
)
_v17_add_teknika_gate(
    "Teknika format: English, 6-10 A4 pages, two columns",
    "MANUSCRIPT_CHECK_REQUIRED",
    "Verify the final Word manuscript against the current Teknika template. Notebook outputs do not establish page count or layout compliance."
)
_v17_add_teknika_gate(
    "Teknika abstract and keywords",
    "MANUSCRIPT_CHECK_REQUIRED",
    "The journal guideline specifies an abstract of at most 250 words and up to five keywords. Verify these in the final English manuscript."
)
_v17_add_teknika_gate(
    "Teknika double-blind review preparation",
    "MANUSCRIPT_CHECK_REQUIRED",
    "Prepare an anonymized review file and move identifying author information to the designated submission metadata or title page as required by the current system."
)
_v17_add_teknika_gate(
    "Teknika similarity index below 25 percent",
    "EXTERNAL_CHECK_REQUIRED",
    "The journal guideline states a similarity index below 25 percent. This cannot be verified by this notebook; run a reputable similarity checker and interpret matches manually."
)
_v17_add_teknika_gate(
    "Teknika generative-AI disclosure",
    "AUTHOR_DECLARATION_REQUIRED",
    "If generative AI assisted drafting, editing, or translation, follow the journal's current AI policy and add the required disclosure. Authors remain responsible for accuracy."
)
_v17_add_teknika_gate(
    "Teknika call-for-papers deadline for Vol. 15 No. 3, November 2026",
    "DEADLINE_PASSED",
    "The posted call lists 30 September 2026 as the deadline. As of 18 October 2026, confirm the next active call or general-submission window with the journal."
)
_v17_add_teknika_gate(
    "Teknika publication charge",
    "CONFIRM_BEFORE_SUBMISSION",
    "The journal announcement dated 10 August 2026 states an APC of IDR 1,250,000 effective from Vol. 15 No. 3 (November 2026), with an optional non-refundable fast-track fee. Confirm current terms directly."
)

# A conservative readiness gate: journal scope match alone is not sufficient.
_v17_required_evidence = bool(
    isinstance(prospective_metrics, pd.DataFrame)
    and not prospective_metrics.empty
    and str(future_gate_status).upper() == "PASS"
    and bool(future_signature_ok)
)
_v17_add_teknika_gate(
    "Teknika readiness decision",
    "NOT_READY" if not _v17_required_evidence else "EVIDENCE_REVIEW_REQUIRED",
    "Scope alignment is promising, but submission readiness still depends on a qualifying frozen-model temporal holdout, label validity, leakage audit, complete station mapping, novelty review, and compliance checks in the final manuscript. The notebook cannot guarantee acceptance."
)


## D. Exactly ten CSV and ten accessible scientific PNG

CSV output tables contain the frozen input signature, primary retrospective model scores, cluster-aware uncertainty, corrected at-risk report-onset scores, day-specific alarm budgets, station-level paired missingness simulations, region-specific support limits, and transparent editorial readiness checks.

PNG artwork has **English scientific labels**, colourblind-aware hues together with line types/markers, and **300 DPI** for single panels or **600 DPI** for composite graphics. The final ZIP contains only these 20 files under one `research_outputs/` directory; model caches and intermediate evidence stay outside it. Scientific captions are kept in the notebook; there are no decorative figure titles embedded into the plots.

In [ ]:
# Tulis tepat sepuluh file tabel berbahasa Inggris dalam UTF-8 dengan kode-kolom yang stabil.
metrics_export=model_metrics.copy()
if not prospective_metrics.empty:
    metrics_export=pd.concat([metrics_export,prospective_metrics],ignore_index=True,sort=False)
CSV_DATA={
 '01_input_integrity_manifest.csv':integrity,
 '02_region_year_observation_quality.csv':quality_audit,
 '03_region_test_quality_and_support.csv':regions_table,
 '04_frozen_test_model_metrics.csv':metrics_export,
 '05_paired_block_bootstrap_sensitivity.csv':bootstrap_table,
 '06_corrected_at_risk_report_onset.csv':onset_table,
 '07_daily_alert_capacity.csv':capacity_table,
 '08_paired_missingness_robustness.csv':robustness_summary,
 '09_date_block_calibration_uncertainty.csv':calibration_table,
 '10_editorial_evidence_gates.csv':editorial_table}
for filename,table in CSV_DATA.items():table.to_csv(OUTPUT/filename,index=False,encoding='utf-8-sig')
assert len(list(OUTPUT.glob('*.csv')))==10
print('Saved 10 CSV tables.')

In [ ]:
# Desain ilustrasi menggunakan warna Okabe-Ito serta garis berbeda untuk aksesibilitas.
def save_plot(fig,name,dpi=600):
    fig.savefig(OUTPUT/name,dpi=(72 if FAST_TEST else dpi),bbox_inches='tight',facecolor='white',pil_kwargs={'compress_level':2})
    plt.close(fig)

focus=['Seasonal-region baseline','HistGradientBoosting','Standard stacking',
       'Full uncertainty–quality stacking','Stress-trained uncertainty–quality stacking']
style={
 'Seasonal-region baseline':(COLORS['black'],':','x'),
 'HistGradientBoosting':(COLORS['green'],'-.','^'),
 'Standard stacking':(COLORS['purple'],'--','D'),
 'Full uncertainty–quality stacking':(COLORS['red'],':','P'),
 'Stress-trained uncertainty–quality stacking':(COLORS['blue'],'-','v')}
# Figure 01 — lokasi stasiun dan kehilangan observasi.
st=frames['station'];st=st[st.in_rainfall_matrix.astype(str).str.lower().eq('true')].copy()
st=st.dropna(subset=['latitude','longitude','metadata_missing_pct'])
fig,ax=plt.subplots(figsize=(6.8,4.2))
scatter=ax.scatter(st.longitude,st.latitude,c=st.metadata_missing_pct,cmap='cividis',s=18,
                   vmin=0,vmax=max(10,float(st.metadata_missing_pct.quantile(.98))),
                   edgecolors='black',linewidths=.18)
fig.colorbar(scatter,ax=ax,label='Metadata-reported missing data (%)')
ax.set(xlabel='Longitude (°E)',ylabel='Latitude (°S)');ax.grid(alpha=.18,linestyle=':')
save_plot(fig,'01_station_quality_geography.png')

# Figure 02 — cakupan observasi tahunan dan prevalensi laporan positif.
audit_year=quality_audit.groupby('year',as_index=False).agg(
    region_days=('region_days','sum'),positives=('positive_region_days','sum'),
    mean_coverage=('mean_available_station_fraction_lag1','mean'))
fig,axes=plt.subplots(1,2,figsize=(6.8,3.6))
axes[0].plot(audit_year.year,audit_year.mean_coverage*100,color=COLORS['blue'],marker='o',lw=1.8)
axes[0].set(xlabel='Calendar year',ylabel='Mean station coverage (%)')
axes[1].plot(audit_year.year,audit_year.positives/audit_year.region_days*100,
             color=COLORS['orange'],ls='--',marker='s',lw=1.8)
axes[1].set(xlabel='Calendar year',ylabel='Recorded-flood region-days (%)')
for ax in axes:ax.grid(alpha=.2,linestyle=':')
save_plot(fig,'02_temporal_coverage_and_reporting.png',600)

# Figure 03 — bandingkan PR pada 2025 yang eksploratif dan periode pasca-2025 bila tersedia.
# AP selalu dihitung pada seluruh observasi dalam periode masing-masing, bukan hanya area plot yang diperbesar.
if (not prospective_metrics.empty) and np.unique(prospective_predictions['flood_recorded'].to_numpy(dtype=int)).size==2:
    fig,axes=plt.subplots(1,2,figsize=(6.8,4.45))
    Y_future_plot=prospective_predictions['flood_recorded'].to_numpy(dtype=int)
    for name in focus[1:]:
        color,ls,marker=style[name]
        precision,recall,_=precision_recall_curve(Y,P[name])
        axes[0].plot(recall,precision,color=color,ls=ls,lw=1.4,
                     label=f'{name}: AP {average_precision_score(Y,P[name]):.3f}')
        future_col=MODELS[name]
        p_future_plot=prospective_predictions[future_col].to_numpy(dtype=float)
        precision_f,recall_f,_=precision_recall_curve(Y_future_plot,p_future_plot)
        axes[1].plot(recall_f,precision_f,color=color,ls=ls,lw=1.4,
                     label=f'{name}: AP {average_precision_score(Y_future_plot,p_future_plot):.3f}')
    axes[0].axhline(Y.mean(),color=COLORS['black'],ls=':',lw=1.3,label=f'Prevalence: {Y.mean():.3f}')
    axes[1].axhline(Y_future_plot.mean(),color=COLORS['black'],ls=':',lw=1.3,
                    label=f'Prevalence: {Y_future_plot.mean():.3f}')
    axes[0].set_title('2025 exploratory test',fontsize=9)
    axes[1].set_title('Post-2025 temporal holdout candidate',fontsize=9)
    for ax in axes:
        ax.set(xlabel='Recall',ylabel='Precision',xlim=(0,1),ylim=(0,1))
        ax.grid(alpha=.18,linestyle=':')
    for ax,tag in zip(axes,['(a)','(b)']):
        ax.text(.025,.97,tag,transform=ax.transAxes,va='top',ha='left',fontsize=10,fontweight='bold')
    h,l=axes[0].get_legend_handles_labels()
    fig.legend(h,l,loc='lower center',bbox_to_anchor=(.5,-.075),ncol=2,fontsize=6.5,frameon=False)
    fig.subplots_adjust(bottom=.32,wspace=.25)
else:
    # Bila data independen belum tersedia, tampilkan kurva lengkap dan detail 2025 tanpa menyiratkan validasi baru.
    fig,axes=plt.subplots(1,2,figsize=(6.8,4.45),gridspec_kw={'width_ratios':[1,1]})
    for name in focus[1:]:
        precision,recall,_=precision_recall_curve(Y,P[name])
        color,ls,marker=style[name]
        legend_label=f'{name}: AP {average_precision_score(Y,P[name]):.3f}'
        for ax in axes:
            ax.plot(recall,precision,color=color,ls=ls,lw=1.4,label=legend_label)
    for ax in axes:
        ax.axhline(Y.mean(),color=COLORS['black'],ls=':',lw=1.3,label=f'Prevalence: {Y.mean():.3f}')
        ax.set(xlabel='Recall',ylabel='Precision')
        ax.grid(alpha=.18,linestyle=':')
    axes[0].set(xlim=(0,1),ylim=(0,1),title='2025 exploratory test')
    axes[1].set(xlim=(0,.55),ylim=(0,.4),title='Detail at lower recall')
    for ax,tag in zip(axes,['(a)','(b)']):
        ax.text(.025,.97,tag,transform=ax.transAxes,va='top',ha='left',fontsize=10,fontweight='bold')
    h,l=axes[0].get_legend_handles_labels()
    fig.legend(h,l,loc='lower center',bbox_to_anchor=(.5,-.075),ncol=2,fontsize=7,frameon=False)
    fig.subplots_adjust(bottom=.27,wspace=.25)
save_plot(fig,'03_precision_recall_2025_exploratory.png',600)

# Figure 04 — AP vs Brier skill score. A baseline-referenced scale avoids exaggerating tiny raw Brier differences.
fig,ax=plt.subplots(figsize=(6.8,4.4))
for name in focus:
    row=model_metrics[model_metrics.model.eq(name)].iloc[0]
    color,_,marker=style[name]
    bss=100*float(row.brier_skill_vs_seasonal_baseline)
    if marker=='x':
        ax.scatter(row.average_precision,bss,s=85,color=color,marker=marker)
    else:
        ax.scatter(row.average_precision,bss,s=85,color=color,edgecolor='black',marker=marker)
    ax.annotate(name.replace(' uncertainty–quality stacking',''),
                (row.average_precision,bss),xytext=(4,4),textcoords='offset points',fontsize=6)
ax.axhline(0,color=COLORS['black'],ls=':',lw=1.1,label='Seasonal baseline (BSS = 0)')
ax.set(xlabel='Average precision (higher is better)',
       ylabel='Brier skill vs seasonal baseline (%) (higher is better)')
ax.set_ylim(bottom=0)
ax.text(.02,.98,'Dashed line: seasonal baseline (BSS = 0)',transform=ax.transAxes,
        va='top',fontsize=7)
ax.grid(alpha=.20,linestyle=':')
save_plot(fig,'04_discrimination_calibration_tradeoff.png')

# Figure 05 — paired AP/Brier differences using non-circular moving blocks of 7/14/30 days.
compare=bootstrap_table[(bootstrap_table.comparison.eq('Stress vs standard')) & (bootstrap_table.evaluation_period.eq('exploratory_2025'))].sort_values('block_days')
fig,axes=plt.subplots(1,2,figsize=(6.8,3.8))
for ax,point,lo,hi,xlabel in [
 (axes[0],'observed_delta_ap','ap_ci95_low','ap_ci95_high','Average precision difference'),
 (axes[1],'observed_brier_improvement','brier_ci95_low','brier_ci95_high','Brier improvement')]:
    xs=compare[point].to_numpy();lower=compare[lo].to_numpy();upper=compare[hi].to_numpy()
    ax.errorbar(xs,compare.block_days,xerr=[xs-lower,upper-xs],fmt='o',color=COLORS['blue'],capsize=4)
    ax.axvline(0,color=COLORS['black'],ls=':');ax.set(xlabel=xlabel,ylabel='Date-block length (days)')
    ax.set_yticks(list(BLOCK_LENGTHS));ax.grid(alpha=.18,linestyle=':')
save_plot(fig,'05_bootstrap_block_length_sensitivity.png',600)

# Figure 06 — AP lift pada estimand asli dan estimand conditional at-risk (prevalensi berbeda).
subset=onset_table[onset_table.model.isin(focus)].copy()
xx=np.arange(len(subset));w=.37
fig,ax=plt.subplots(figsize=(6.8,4.5))
ax.bar(xx-w/2,subset.original_report_day_ap/Y.mean(),width=w,label='Reported-flood day AP / prevalence',color=COLORS['blue'])
ax.bar(xx+w/2,subset.onset_at_risk_ap/subset.onset_at_risk_prevalence,width=w,
       label='Eligible 7-day onset AP / at-risk prevalence',color=COLORS['orange'],hatch='//')
ax.axhline(1,color=COLORS['black'],ls=':',lw=1.1,label='Prevalence-level benchmark')
ax.set_xticks(xx)
ax.set_xticklabels([str(q).replace(' uncertainty–quality stacking','') for q in subset.model],rotation=15,ha='right')
ax.set(ylabel='Average precision lift over target prevalence')
ax.legend(frameon=False,fontsize=7)
ax.grid(axis='y',alpha=.18,linestyle=':')
save_plot(fig,'06_corrected_at_risk_report_onset.png')

# Figure 07 — harian satu/dua/tiga wilayah, manajemen kapasitas alert.
fig,ax=plt.subplots(figsize=(6.8,4.5))
for name in focus:
    sub=capacity_table[capacity_table.model.eq(name)].sort_values('regions_flagged_per_day')
    color,ls,marker=style[name]
    ax.plot(sub.regions_flagged_per_day,sub.recall*100,marker=marker,color=color,ls=ls,lw=1.7,label=name)
ax.set(xlabel='Regions flagged per day',ylabel='Recorded-flood region-days captured (%)')
ax.set_xticks([1,2,3]);ax.legend(frameon=False,fontsize=7);ax.grid(alpha=.18,linestyle=':')
save_plot(fig,'07_daily_alert_capacity.png')

# Figure 08 — kalibrasi dengan interval bootstrap blok 7 hari (bukan IID Wilson).
fig,ax=plt.subplots(figsize=(6.8,4.8))
for name in ['HistGradientBoosting','Standard stacking','Stress-trained uncertainty–quality stacking']:
    sub=calibration_table[calibration_table.model.eq(name)].sort_values('bin')
    color,ls,marker=style[name]
    x=sub.mean_probability.to_numpy(dtype=float)
    v=sub.observed_report_fraction.to_numpy(dtype=float)
    lo=sub.date_block7_boot95_low.to_numpy(dtype=float)
    hi=sub.date_block7_boot95_high.to_numpy(dtype=float)
    ax.errorbar(x,v,yerr=[np.maximum(0,v-lo),np.maximum(0,hi-v)],fmt=marker,ls=ls,color=color,
                lw=1.2,capsize=2,label=name)
ax.plot([0,.16],[0,.16],ls=':',color=COLORS['black'],label='Perfect calibration')
ax.set(xlim=(0,.16),ylim=(0,.19),xlabel='Mean predicted probability',ylabel='Observed report frequency')
ax.legend(frameon=False,fontsize=7)
ax.grid(alpha=.18,linestyle=':')
save_plot(fig,'08_reliability_date_block_bootstrap.png',600)

# Figure 09 — dampak missingness dalam AP dan Brier, dibedakan dengan mekanisme dan garis.
fig,axes=plt.subplots(1,2,figsize=(6.8,4.1))
mechs=['MCAR','Station outage','Temporal block','Quality-dependent']
colors=[COLORS['blue'],COLORS['orange'],COLORS['green'],COLORS['purple']]
for mech,color,mark,ls in zip(mechs,colors,['o','s','^','D'],['-','--','-.',':']):
    sub=robustness_summary[(robustness_summary.mechanism.eq(mech))&
                            (robustness_summary.comparator.eq('Standard stacking'))].sort_values('additional_missing_target_fraction')
    x=sub.additional_missing_target_fraction.to_numpy()*100
    axes[0].plot(x,sub.delta_ap_mean,color=color,marker=mark,ls=ls,lw=1.5,label=mech)
    axes[1].plot(x,sub.brier_improvement_mean*1e4,color=color,marker=mark,ls=ls,lw=1.5,label=mech)
axes[0].set(xlabel='Additional missing station-days (%)',ylabel='AP: stress-trained − standard')
axes[1].set(xlabel='Additional missing station-days (%)',ylabel='Brier improvement × 10⁴')
for ax in axes:ax.axhline(0,color=COLORS['black'],ls=':');ax.grid(alpha=.18,linestyle=':')
axes[0].legend(frameon=False,fontsize=7)
save_plot(fig,'09_paired_station_missingness_tradeoff.png',600)

# Figure 10 — jumlah positif untuk evaluasi per wilayah dan ketersediaan data.
r=regions_table.sort_values('test_positive_n',ascending=True)
fig,axes=plt.subplots(1,2,figsize=(6.8,6.1),sharey=True)
idx=np.arange(len(r))
colors=[COLORS['green'] if n>=10 else COLORS['gray'] for n in r.test_positive_n]
axes[0].barh(idx,r.test_positive_n,color=colors,edgecolor='black',linewidth=.2)
axes[0].axvline(10,ls='--',color=COLORS['red'],label='Heuristic display threshold (10 positives)')
axes[0].set(xlabel='Recorded positives in test year',yticks=idx,yticklabels=r.region_key)
axes[0].legend(frameon=False,fontsize=7)
missing_pct=(1-r.mean_lag1_station_coverage)*100
axes[1].barh(idx,missing_pct,color=COLORS['sky'],edgecolor='black',linewidth=.2)
axes[1].set(xlabel='Missing station-days in 2025 (%)',xlim=(0,5));axes[1].tick_params(labelleft=False)
for ax in axes:ax.grid(axis='x',alpha=.16,linestyle=':')
save_plot(fig,'10_regional_support_and_missingness.png',600)
print('Saved 10 PNG figures.')

## E. Interpretasi ilmiah dan keputusan publikasi

**Status hasil:** eksploratif. Evaluasi utama memakai laporan banjir wilayah-hari pada 2025, periode yang telah diperiksa untuk membandingkan model dan menyusun analisis sensitivitas. Karena itu, angka 2025 tidak boleh disebut sebagai uji konfirmatori yang sepenuhnya independen.

**Temuan yang dapat dilaporkan secara terbatas:** stress-trained uncertainty–quality stacking memperoleh AP 0,1062, ROC-AUC 0,7575, dan Brier score 0,028106 pada periode eksploratif 2025. Keunggulan AP terhadap HistGradientBoosting dan standard stacking tidak konsisten pada semua panjang blok dan interval yang disesuaikan untuk beberapa perbandingan. Peningkatan Brier lebih konsisten, tetapi besarnya masih kecil.

**Klaim universal tentang ketahanan AP terhadap missingness tidak didukung.** Pada eksperimen paired missingness, stress-trained stacking umumnya tidak mengungguli standard stacking dari sisi AP. Naskah harus melaporkan trade-off ini secara eksplisit. Jangan memilih skenario, ambang, atau model berdasarkan hasil 2025 lalu menyebut hasil yang sama sebagai validasi independen.

**Definisi outcome:** label merepresentasikan laporan banjir, bukan inventaris hidrologis yang membuktikan semua kejadian banjir maupun ketiadaan banjir. Istilah yang aman adalah *recorded flood-report prediction*. Jangan mengklaim peringatan banjir 24 jam sebelum kejadian karena waktu onset fisik belum diverifikasi.

**Syarat sebelum submit:** (1) dokumentasikan waktu ketersediaan setiap fitur pada saat prediksi dan tanggal pelaporan banjir; (2) selesaikan atau jelaskan dua dari 460 header stasiun yang belum terpetakan; (3) lakukan validasi temporal independen setelah model dibekukan, jika label baru tersedia; (4) bandingkan novelty dengan literatur terbaru secara sistematis; (5) arsipkan notebook, versi data, model, lingkungan, kamus data, dan output; serta (6) tetapkan biaya false alarm dan missed event sebelum membuat klaim operasional.

**Keputusan publikasi:** belum siap submit. Kualitas kode dan banyaknya eksperimen tidak menggantikan validasi independen, definisi outcome yang kuat, dan novelty yang teruji terhadap literatur.

**Reproducibility rule:** jangan mengubah ambang atau pengaturan model setelah memeriksa skor 2025. Jika spesifikasi model berubah, perbarui versi pipeline dan tanda tangan model, lalu dokumentasikan perubahan. Hasil 2025 tetap eksploratif.

**Captions:** (1) Station geography and metadata-reported missingness. (2) Temporal coverage and reporting prevalence. (3) Precision–recall curves for the exploratory 2025 period. (4) Discrimination versus Brier skill. (5) Paired date-block uncertainty. (6) Corrected at-risk report-onset sensitivity. (7) Retrospective daily alert capacity. (8) Date-block calibration reliability. (9) Paired missingness effects on AP and Brier score. (10) Regional positive support and observation coverage.


In [ ]:
# Final: verifikasi artefak dan unduh satu ZIP melalui API resmi Python Colab.
# Fast-test outputs are intentionally blocked from being labelled publication-ready.
from PIL import Image
if FAST_TEST:
    raise RuntimeError('FAST_TEST=True menghasilkan artefak diagnostik, bukan output publikasi. Set FAST_TEST=False dan jalankan ulang seluruh notebook.')
pngs=sorted(OUTPUT.glob('*.png'));csvs=sorted(OUTPUT.glob('*.csv'))
if len(pngs)!=10 or len(csvs)!=10:
    raise RuntimeError(f'Output should be 10 PNG + 10 CSV; observed {len(pngs)} and {len(csvs)}.')
if len(list(OUTPUT.iterdir()))!=20:
    raise RuntimeError('Unexpected files in the publication output folder.')
for image_path in pngs:
    with Image.open(image_path) as im:
        im.verify()
    with Image.open(image_path) as im:
        dpi=im.info.get('dpi',(0,0))
        if not FAST_TEST and min(dpi)<295:
            raise ValueError(f'Image resolution below 300 DPI: {image_path} {dpi}')
ZIP_OUT=ROOT/'flood_v18_teknika_sinta2_10_PNG_10_CSV.zip'
with zipfile.ZipFile(ZIP_OUT,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3) as archive:
    for f in pngs+csvs: archive.write(f,arcname='research_outputs/'+f.name)
with zipfile.ZipFile(ZIP_OUT) as archive:
    if archive.testzip() is not None:raise RuntimeError('ZIP integrity check failed')
    names=archive.namelist()
    assert len(names)==20 and len(set(names))==20
    assert all(n.startswith('research_outputs/') for n in names)
    assert sum(n.endswith('.png') for n in names)==10
    assert sum(n.endswith('.csv') for n in names)==10
print('ZIP verified:',ZIP_OUT,'| Files:',len(names),'| Bytes:',ZIP_OUT.stat().st_size)
print('Scientific caution: results use a repeatedly reviewed 2025 exploratory test, not a new independent holdout.')
if IN_COLAB:
    try:
        from google.colab import files
        LOCAL_ZIP_OUT=Path('/content')/ZIP_OUT.name
        shutil.copy2(ZIP_OUT,LOCAL_ZIP_OUT)
        if not LOCAL_ZIP_OUT.exists() or LOCAL_ZIP_OUT.stat().st_size!=ZIP_OUT.stat().st_size:
            raise IOError('Local download copy failed verification.')
        files.download(str(LOCAL_ZIP_OUT))  # Official Python Colab API; avoid a browser-side JavaScript download helper.
        print('Verified ZIP retained in project output location:',ZIP_OUT)
    except Exception as exc:
        print('Automatic browser download did not complete. The verified ZIP remains saved at:',ZIP_OUT)
        print('Download diagnostic:',repr(exc))
else:
    print('Local verified ZIP:',ZIP_OUT)

## Optional archival export of existing original-run files
Run this final cell only after mounting the original Drive. It does not retrain.
Current captured runtime does not prove the historical training environment. Missing files remain documented.


In [ ]:
# Ekspor berkas asli tanpa menjalankan ulang seluruh eksperimen.
export_source = '#!/usr/bin/env python3\n"""Ekspor berkas run yang benar-benar ada. Tidak melatih atau memuat model pickle."""\nimport argparse\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport shutil\nimport sys\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nEXPECTED_EVIDENCE_SHA = \'2a476134a9b27dcf28639574dcdd68b3b5580a951cd37763fe3e5a95706c1d72\'\nSOURCES = {\n \'Data-Ch-Jabar-2020-2025.xlsx\':\'92a207057853ed063fa4548892e515ae7d160f582a4469cde20f268e4230c99f\',\n \'Dataset_Flood_2020-2025.xlsx\':\'a5265db438bac4bd49d4a27d5dedba2d8beca405c49a5644630f6292858f3c28\',\n \'Dataset_Rainfall_Floods_2020-2025.xlsx\':\'c873bf2fa9a503f599db803325137ccf6149532254550517323ac9e6c4ae1d89\',\n \'West_Java_Rainfall_Station_Metadata_Lengkap.xlsx\':\'15afa28cec99f00825f97e7b5915d1a5caff7d16eb84851fe2d3a616c694d89a\'}\nSTEMS = {\n \'station\':\'01_station_mapping_audit\',\'quality\':\'02_region_year_quality\',\n \'panel\':\'03_lagged_feature_panel\',\'split\':\'04_temporal_split_balance\',\n \'metrics\':\'05_model_metrics_validation_and_test\',\'pred\':\'06_held_out_test_predictions\',\n \'calibration\':\'07_test_reliability_bins\',\'robustness\':\'08_station_missingness_robustness\',\n \'alerts\':\'09_operational_alert_budgets\',\'bootstrap\':\'10_paired_block_bootstrap\'}\n\ndef sha(path):\n    h=hashlib.sha256()\n    with path.open(\'rb\') as stream:\n        for chunk in iter(lambda:stream.read(1024*1024),b\'\'): h.update(chunk)\n    return h.hexdigest()\n\ndef export(project_root, output):\n    root=Path(project_root).expanduser().resolve();out=Path(output).expanduser().resolve()\n    if not root.is_dir(): raise ValueError(\'Project root does not exist; mount the original Drive first.\')\n    if out.exists() or out.with_suffix(\'.zip\').exists(): raise ValueError(\'Output path or ZIP already exists; choose a new destination.\')\n    if out==root or out.is_relative_to(root) or root.is_relative_to(out):\n        raise ValueError(\'Use an output directory outside the source project tree.\')\n    records=[]; missing=[]; todo=[]\n    for name,expected in SOURCES.items():\n        p=root/\'source\'/name\n        if not p.is_file():missing.append(\'source/\'+name);continue\n        actual=sha(p)\n        if actual!=expected:raise ValueError(\'Source hash mismatch: \'+name+\'; preserve as a separate new run.\')\n        todo.append((p,\'data/source/\'+name))\n    evidence={key:root/\'v15_training_evidence_cache\'/(stem+\'.csv\') for key,stem in STEMS.items()}\n    # Jangan mengekspor potongan cache seolah-olah satu set lengkap.\n    if all(p.is_file() for p in evidence.values()):\n        combined=hashlib.sha256(\'\'.join(sha(evidence[key]) for key in sorted(evidence)).encode()).hexdigest()\n        if combined!=EXPECTED_EVIDENCE_SHA:\n            raise ValueError(\'Training evidence signature differs from the supplied historical record; export this new run separately.\')\n        todo += [(p,\'training_evidence/\'+p.name) for p in evidence.values()]\n        evidence_state=\'all ten files match recorded historical combined SHA-256\'\n    else:\n        missing += [\'training_evidence/\'+p.name for p in evidence.values() if not p.is_file()]\n        evidence_state=\'incomplete; no partial evidence set copied\'\n    bundle=root/\'cache\'/\'frozen_model_bundle_fe8bc39e393c2c1d.joblib\'\n    if bundle.is_file():todo.append((bundle,\'artifacts/\'+bundle.name))\n    else:missing.append(\'artifacts/\'+bundle.name)\n    out.mkdir(parents=True)\n    for original,relative in todo:\n        dest=out/relative;dest.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(original,dest)\n        records.append({\'path\':relative,\'sha256\':sha(dest),\'size_bytes\':dest.stat().st_size})\n    environment=out/\'environment\';environment.mkdir()\n    pins=sorted(set(f"{d.metadata[\'Name\']}=={d.version}" for d in importlib.metadata.distributions() if d.metadata.get(\'Name\')))\n    (environment/\'requirements-captured.txt\').write_text(\'\\n\'.join(pins)+\'\\n\')\n    runtime={\'captured_at_utc\':datetime.now(timezone.utc).isoformat(),\'python\':sys.version,\n             \'platform\':platform.platform(),\'historical_runtime_verified\':False,\n             \'qualification\':\'Current runtime at capture; does not prove original training environment.\'}\n    (environment/\'runtime-captured.json\').write_text(json.dumps(runtime,indent=2)+\'\\n\')\n    status={\'missing\':missing,\'training_evidence\':evidence_state,\n            \'model_bundle_copied\':bundle.is_file(),\'model_metadata_independently_verified\':False,\n            \'qualification\':\'Binary copied without unpickling; filename alone does not authenticate its internal signature.\',\n            \'historical_runtime_verified\':False,\'end_to_end_reproduction_verified\':False,\n            \'source_data_attribution\':\'Preserve original dataset creators, DOI 10.17632/d7j3s64vvr.1, and CC BY 4.0 attribution.\'}\n    (out/\'export_status.json\').write_text(json.dumps(status,indent=2)+\'\\n\')\n    # Manifest mencakup environment/status juga; manifest tidak menghash dirinya sendiri.\n    records=[{\'path\':str(p.relative_to(out)),\'sha256\':sha(p),\'size_bytes\':p.stat().st_size}\n             for p in sorted(out.rglob(\'*\')) if p.is_file()]\n    (out/\'export_file_manifest.json\').write_text(json.dumps(records,indent=2)+\'\\n\')\n    archive=out.with_suffix(\'.zip\')\n    with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(out.rglob(\'*\')):\n            if p.is_file():z.write(p,str(Path(out.name)/p.relative_to(out)))\n    print(json.dumps({\'archive\':str(archive),\'missing_count\':len(missing),\'status\':status},indent=2))\n    return archive\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--project-root\',required=True);parser.add_argument(\'--output\',required=True)\n    args=parser.parse_args()\n    export(args.project_root,args.output)\n'
archive_namespace = {"__name__": "archive_export"}
exec(compile(export_source, "export_run_record.py", "exec"), archive_namespace)
from pathlib import Path
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
archive = archive_namespace["export"]("/content/drive/MyDrive/Flood_Research_Mendeley_v1", "/content/flood_v18_supplemental_record")
from google.colab import files
files.download(str(archive))
